In [ ]:
from google.colab import drive
import zipfile
import os

# Mount Google Drive
drive.mount('/content/drive')

# Path to your ZIP in Drive
zip_path = "/content/drive/My Drive/diabeties.zip"

# Extract it
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("extracted_files")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import os
from glob import glob
from tqdm import tqdm
import zipfile
import warnings
warnings.filterwarnings('ignore')

# TensorFlow/Keras imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import EfficientNetB3, ResNet50V2, DenseNet121
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    Dense, Dropout, GlobalAveragePooling2D, BatchNormalization,
    Conv2D, MaxPooling2D, Flatten, Input, Concatenate, Add
)
from tensorflow.keras.optimizers import Adam, SGD, RMSprop
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau,
    LearningRateScheduler, TensorBoard, CSVLogger
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.regularizers import l2
from tensorflow.keras import backend as K

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.utils.class_weight import compute_class_weight
import pickle

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# GPU Configuration
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    for device in physical_devices:
        tf.config.experimental.set_memory_growth(device, True)
    print(f"✓ GPU Available: {len(physical_devices)} device(s)")
else:
    print("⚠ No GPU found, using CPU")

# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
except:
    print("Not running in Colab")

# Configuration
CONFIG = {
    # Data
    'IMG_SIZE': (224, 224),
    'DATA_DIR': '/content/diabeties',
    'ZIP_PATH': '/content/drive/MyDrive/diabeties.zip',
    'BATCH_SIZE': 32,
    'NUM_CLASSES': 5,
    'RANDOM_STATE': 42,

    # Model Architecture
    'BASE_MODEL': 'EfficientNetB3',  # Options: EfficientNetB3, ResNet50V2, DenseNet121, Custom
    'TRAINABLE_LAYERS': 50,  # Number of layers to unfreeze for fine-tuning
    'DROPOUT_RATE': 0.3,
    'L2_REGULARIZATION': 0.0001,

    # Training Strategy
    'EPOCHS': 100,
    'INITIAL_LEARNING_RATE': 0.001,
    'USE_MIXED_PRECISION': True,  # Faster training on modern GPUs

    # Learning Rate Schedule
    'LR_SCHEDULE': 'cosine_decay_restart',  # Options: reduce_on_plateau, cosine_decay, cosine_decay_restart, exponential, step_decay, custom

    # Callbacks
    'EARLY_STOPPING_PATIENCE': 15,
    'REDUCE_LR_PATIENCE': 5,
    'REDUCE_LR_FACTOR': 0.5,
    'MIN_LR': 1e-7,

    # Data Augmentation
    'USE_AUGMENTATION': True,
    'AUGMENTATION_STRENGTH': 'medium',  # light, medium, strong

    # Advanced Training Techniques
    'USE_CLASS_WEIGHTS': True,
    'USE_FOCAL_LOSS': True,  # Better for imbalanced data
    'USE_LABEL_SMOOTHING': 0.1,  # Prevents overconfidence
    'USE_MIXUP': True,  # Data augmentation in feature space
    'MIXUP_ALPHA': 0.2,
    'USE_GRADIENT_CLIPPING': True,
    'GRADIENT_CLIP_VALUE': 1.0,

    # Two-Stage Training
    'USE_TWO_STAGE': True,
    'STAGE1_EPOCHS': 20,  # Train only top layers
    'STAGE2_EPOCHS': 80,   # Fine-tune entire model

    'CLASS_NAMES': {
        0: 'No DR',
        1: 'Mild',
        2: 'Moderate',
        3: 'Severe',
        4: 'Proliferative DR'
    }
}

# Enable mixed precision for faster training
if CONFIG['USE_MIXED_PRECISION']:
    policy = tf.keras.mixed_precision.Policy('mixed_float16')
    tf.keras.mixed_precision.set_global_policy(policy)
    print("✓ Mixed precision enabled")


# Custom Learning Rate Schedulers
class CustomLRScheduler:
    """Collection of learning rate schedules"""

    @staticmethod
    def cosine_decay_with_warmup(epoch, total_epochs, initial_lr, warmup_epochs=5):
        """Cosine decay with warmup"""
        if epoch < warmup_epochs:
            return initial_lr * (epoch + 1) / warmup_epochs

        progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
        return initial_lr * 0.5 * (1 + np.cos(np.pi * progress))

    @staticmethod
    def cosine_decay_with_restart(epoch, initial_lr, restart_epochs=20):
        """Cosine annealing with warm restarts"""
        cycle = np.floor(1 + epoch / restart_epochs)
        x = np.abs(epoch / restart_epochs - cycle)
        return initial_lr * 0.5 * (1 + np.cos(np.pi * x))

    @staticmethod
    def exponential_decay(epoch, initial_lr, decay_rate=0.95):
        """Exponential decay"""
        return initial_lr * (decay_rate ** epoch)

    @staticmethod
    def step_decay(epoch, initial_lr, drop_rate=0.5, epochs_drop=10):
        """Step decay"""
        return initial_lr * (drop_rate ** np.floor(epoch / epochs_drop))


# Custom Losses
class FocalLoss(tf.keras.losses.Loss):
    """Focal Loss for handling class imbalance"""

    def __init__(self, alpha=0.25, gamma=2.0, from_logits=False):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.from_logits = from_logits

    def call(self, y_true, y_pred):
        if self.from_logits:
            y_pred = tf.nn.softmax(y_pred, axis=-1)

        # Clip predictions to prevent log(0)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)

        # Calculate cross entropy
        ce = -y_true * tf.math.log(y_pred)

        # Calculate focal loss
        weight = self.alpha * y_true * tf.pow(1 - y_pred, self.gamma)
        focal_loss = weight * ce

        return tf.reduce_sum(focal_loss, axis=-1)


def mixup_data(x, y, alpha=0.2):
    """Apply mixup augmentation"""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = x.shape[0]
    index = np.random.permutation(batch_size)

    mixed_x = lam * x + (1 - lam) * x[index]
    mixed_y = lam * y + (1 - lam) * y[index]

    return mixed_x, mixed_y


class DeepLearningDRClassifier:
    """Advanced Deep Learning Classifier for Diabetic Retinopathy"""

    def __init__(self, config):
        self.config = config
        self.model = None
        self.history = None
        self.class_weights = None

    def extract_dataset(self):
        """Extract zip file"""
        if os.path.exists(self.config['DATA_DIR']):
            print(f"✓ Dataset already extracted")
            return

        print("Extracting dataset...")
        with zipfile.ZipFile(self.config['ZIP_PATH'], 'r') as zip_ref:
            zip_ref.extractall('/content/')
        print("✓ Extraction complete")

    def load_images_from_split(self, split):
        """Load images and labels"""
        images = []
        labels = []

        split_path = os.path.join(self.config['DATA_DIR'], split)
        if not os.path.exists(split_path):
            return np.array(images), np.array(labels)

        subdirs = sorted([d for d in os.listdir(split_path)
                         if os.path.isdir(os.path.join(split_path, d))])

        for subdir in subdirs:
            class_folder = os.path.join(split_path, subdir)
            class_num = None
            for i in range(5):
                if f'class{i}' in subdir.lower():
                    class_num = i
                    break

            if class_num is None:
                continue

            extensions = ['*.jpg', '*.png', '*.jpeg', '*.JPG', '*.PNG', '*.JPEG']
            image_files = []
            for ext in extensions:
                image_files.extend(glob(os.path.join(class_folder, ext)))

            print(f"  Loading {len(image_files)} images from {split}/{subdir} (Class {class_num})")

            for img_path in tqdm(image_files, desc=f"{split}-{subdir}", leave=False):
                try:
                    img = cv2.imread(img_path)
                    if img is None:
                        continue

                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    img = self._preprocess_image(img)
                    img = cv2.resize(img, self.config['IMG_SIZE'])

                    images.append(img)
                    labels.append(class_num)

                except Exception as e:
                    continue

        return np.array(images), np.array(labels)

    def _preprocess_image(self, img):
        """Advanced preprocessing for retinal images"""
        # Apply CLAHE
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        l = clahe.apply(l)
        img = cv2.merge([l, a, b])
        img = cv2.cvtColor(img, cv2.COLOR_LAB2RGB)

        # Gaussian blur
        img = cv2.GaussianBlur(img, (3, 3), 0)

        return img

    def load_data(self):
        """Load all datasets"""
        print("\n" + "="*70)
        print("LOADING DATASET")
        print("="*70)

        self.X_train, self.y_train = self.load_images_from_split('train')
        self.X_test, self.y_test = self.load_images_from_split('test')
        self.X_val, self.y_val = self.load_images_from_split('val')

        # Normalize images
        self.X_train = self.X_train.astype('float32') / 255.0
        self.X_test = self.X_test.astype('float32') / 255.0
        self.X_val = self.X_val.astype('float32') / 255.0

        # Convert labels to categorical
        self.y_train_cat = tf.keras.utils.to_categorical(self.y_train, self.config['NUM_CLASSES'])
        self.y_test_cat = tf.keras.utils.to_categorical(self.y_test, self.config['NUM_CLASSES'])
        self.y_val_cat = tf.keras.utils.to_categorical(self.y_val, self.config['NUM_CLASSES'])

        print(f"\n✓ Dataset loaded:")
        print(f"  Train: {self.X_train.shape}")
        print(f"  Test:  {self.X_test.shape}")
        print(f"  Val:   {self.X_val.shape}")

        self._print_class_distribution()
        self._calculate_class_weights()

    def _print_class_distribution(self):
        """Print class distribution"""
        print("\n" + "="*70)
        print("CLASS DISTRIBUTION")
        print("="*70)

        for name, labels in [('Train', self.y_train), ('Test', self.y_test), ('Val', self.y_val)]:
            counts = np.bincount(labels, minlength=5)
            total = len(labels)
            print(f"\n{name} ({total} images):")
            for cls, count in enumerate(counts):
                cls_name = self.config['CLASS_NAMES'][cls]
                percentage = (count / total * 100) if total > 0 else 0
                print(f"  {cls_name:20s}: {count:5d} ({percentage:5.1f}%)")

    def _calculate_class_weights(self):
        """Calculate class weights for imbalanced data"""
        if self.config['USE_CLASS_WEIGHTS']:
            classes = np.unique(self.y_train)
            weights = compute_class_weight('balanced', classes=classes, y=self.y_train)
            self.class_weights = dict(zip(classes, weights))

            print("\n" + "="*70)
            print("CLASS WEIGHTS")
            print("="*70)
            for cls, weight in self.class_weights.items():
                print(f"  {self.config['CLASS_NAMES'][cls]:20s}: {weight:.3f}")

    def create_data_generators(self):
        """Create data generators with augmentation"""
        print("\n" + "="*70)
        print("DATA AUGMENTATION")
        print("="*70)

        if self.config['USE_AUGMENTATION']:
            strength = self.config['AUGMENTATION_STRENGTH']

            if strength == 'light':
                aug_params = {
                    'rotation_range': 10,
                    'width_shift_range': 0.1,
                    'height_shift_range': 0.1,
                    'zoom_range': 0.1,
                    'horizontal_flip': True,
                    'fill_mode': 'reflect'
                }
            elif strength == 'medium':
                aug_params = {
                    'rotation_range': 20,
                    'width_shift_range': 0.15,
                    'height_shift_range': 0.15,
                    'shear_range': 0.15,
                    'zoom_range': 0.15,
                    'horizontal_flip': True,
                    'vertical_flip': True,
                    'brightness_range': [0.8, 1.2],
                    'fill_mode': 'reflect'
                }
            else:  # strong
                aug_params = {
                    'rotation_range': 30,
                    'width_shift_range': 0.2,
                    'height_shift_range': 0.2,
                    'shear_range': 0.2,
                    'zoom_range': 0.2,
                    'horizontal_flip': True,
                    'vertical_flip': True,
                    'brightness_range': [0.7, 1.3],
                    'fill_mode': 'reflect'
                }

            print(f"✓ Using {strength} augmentation")
            self.train_datagen = ImageDataGenerator(**aug_params)
        else:
            print("✓ No augmentation")
            self.train_datagen = ImageDataGenerator()

        self.val_datagen = ImageDataGenerator()

    def build_model(self):
        """Build the deep learning model"""
        print("\n" + "="*70)
        print("BUILDING MODEL")
        print("="*70)

        input_shape = (*self.config['IMG_SIZE'], 3)

        if self.config['BASE_MODEL'] == 'EfficientNetB3':
            base_model = EfficientNetB3(
                include_top=False,
                weights='imagenet',
                input_shape=input_shape
            )
        elif self.config['BASE_MODEL'] == 'ResNet50V2':
            base_model = ResNet50V2(
                include_top=False,
                weights='imagenet',
                input_shape=input_shape
            )
        elif self.config['BASE_MODEL'] == 'DenseNet121':
            base_model = DenseNet121(
                include_top=False,
                weights='imagenet',
                input_shape=input_shape
            )
        else:
            raise ValueError(f"Unknown base model: {self.config['BASE_MODEL']}")

        # Freeze base model initially
        base_model.trainable = False

        # Build top layers
        inputs = Input(shape=input_shape)
        x = base_model(inputs, training=False)
        x = GlobalAveragePooling2D()(x)
        x = BatchNormalization()(x)
        x = Dropout(self.config['DROPOUT_RATE'])(x)
        x = Dense(512, activation='relu', kernel_regularizer=l2(self.config['L2_REGULARIZATION']))(x)
        x = BatchNormalization()(x)
        x = Dropout(self.config['DROPOUT_RATE'])(x)
        x = Dense(256, activation='relu', kernel_regularizer=l2(self.config['L2_REGULARIZATION']))(x)
        x = Dropout(self.config['DROPOUT_RATE'] / 2)(x)

        # Output layer
        if self.config['USE_MIXED_PRECISION']:
            outputs = Dense(self.config['NUM_CLASSES'], activation='softmax', dtype='float32')(x)
        else:
            outputs = Dense(self.config['NUM_CLASSES'], activation='softmax')(x)

        self.model = Model(inputs, outputs)

        print(f"✓ Model built: {self.config['BASE_MODEL']}")
        print(f"  Total parameters: {self.model.count_params():,}")
        print(f"  Trainable parameters: {sum([K.count_params(w) for w in self.model.trainable_weights]):,}")

    def compile_model(self, learning_rate=None):
        """Compile the model"""
        if learning_rate is None:
            learning_rate = self.config['INITIAL_LEARNING_RATE']

        # Optimizer with gradient clipping
        if self.config['USE_GRADIENT_CLIPPING']:
            optimizer = Adam(
                learning_rate=learning_rate,
                clipvalue=self.config['GRADIENT_CLIP_VALUE']
            )
        else:
            optimizer = Adam(learning_rate=learning_rate)

        # Loss function
        if self.config['USE_FOCAL_LOSS']:
            loss = FocalLoss(alpha=0.25, gamma=2.0)
            print("✓ Using Focal Loss")
        elif self.config['USE_LABEL_SMOOTHING'] > 0:
            loss = tf.keras.losses.CategoricalCrossentropy(
                label_smoothing=self.config['USE_LABEL_SMOOTHING']
            )
            print(f"✓ Using Label Smoothing: {self.config['USE_LABEL_SMOOTHING']}")
        else:
            loss = 'categorical_crossentropy'

        self.model.compile(
            optimizer=optimizer,
            loss=loss,
            metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
        )

    def create_callbacks(self, stage='stage1'):
        """Create training callbacks"""
        callbacks = []

        # Model checkpoint
        checkpoint = ModelCheckpoint(
            f'best_model_{stage}.h5',
            monitor='val_accuracy',
            save_best_only=True,
            mode='max',
            verbose=1
        )
        callbacks.append(checkpoint)

        # Early stopping
        early_stop = EarlyStopping(
            monitor='val_loss',
            patience=self.config['EARLY_STOPPING_PATIENCE'],
            restore_best_weights=True,
            verbose=1
        )
        callbacks.append(early_stop)

        # Learning rate schedule
        lr_schedule = self.config['LR_SCHEDULE']

        if lr_schedule == 'reduce_on_plateau':
            reduce_lr = ReduceLROnPlateau(
                monitor='val_loss',
                factor=self.config['REDUCE_LR_FACTOR'],
                patience=self.config['REDUCE_LR_PATIENCE'],
                min_lr=self.config['MIN_LR'],
                verbose=1
            )
            callbacks.append(reduce_lr)
            print(f"✓ Using ReduceLROnPlateau")

        elif lr_schedule == 'cosine_decay':
            lr_scheduler = LearningRateScheduler(
                lambda epoch: CustomLRScheduler.cosine_decay_with_warmup(
                    epoch,
                    self.config['EPOCHS'],
                    self.config['INITIAL_LEARNING_RATE']
                )
            )
            callbacks.append(lr_scheduler)
            print(f"✓ Using Cosine Decay with Warmup")

        elif lr_schedule == 'cosine_decay_restart':
            lr_scheduler = LearningRateScheduler(
                lambda epoch: CustomLRScheduler.cosine_decay_with_restart(
                    epoch,
                    self.config['INITIAL_LEARNING_RATE'],
                    restart_epochs=20
                )
            )
            callbacks.append(lr_scheduler)
            print(f"✓ Using Cosine Decay with Restarts")

        elif lr_schedule == 'exponential':
            lr_scheduler = LearningRateScheduler(
                lambda epoch: CustomLRScheduler.exponential_decay(
                    epoch,
                    self.config['INITIAL_LEARNING_RATE']
                )
            )
            callbacks.append(lr_scheduler)
            print(f"✓ Using Exponential Decay")

        elif lr_schedule == 'step_decay':
            lr_scheduler = LearningRateScheduler(
                lambda epoch: CustomLRScheduler.step_decay(
                    epoch,
                    self.config['INITIAL_LEARNING_RATE']
                )
            )
            callbacks.append(lr_scheduler)
            print(f"✓ Using Step Decay")

        # CSV Logger
        csv_logger = CSVLogger(f'training_log_{stage}.csv')
        callbacks.append(csv_logger)

        # TensorBoard
        tensorboard = TensorBoard(
            log_dir=f'./logs/{stage}',
            histogram_freq=0,
            write_graph=False
        )
        callbacks.append(tensorboard)

        return callbacks

    def train_stage1(self):
        """Stage 1: Train only top layers"""
        print("\n" + "="*70)
        print("STAGE 1: TRAINING TOP LAYERS")
        print("="*70)

        self.compile_model()
        callbacks = self.create_callbacks('stage1')

        history = self.model.fit(
            self.train_datagen.flow(
                self.X_train, self.y_train_cat,
                batch_size=self.config['BATCH_SIZE']
            ),
            epochs=self.config['STAGE1_EPOCHS'],
            validation_data=(self.X_val, self.y_val_cat),
            class_weight=self.class_weights,
            callbacks=callbacks,
            verbose=1
        )

        return history

    def train_stage2(self):
        """Stage 2: Fine-tune entire model"""
        print("\n" + "="*70)
        print("STAGE 2: FINE-TUNING ENTIRE MODEL")
        print("="*70)

        # Unfreeze base model layers
        base_model = self.model.layers[1]
        base_model.trainable = True

        # Freeze early layers, train later layers
        trainable_layers = self.config['TRAINABLE_LAYERS']
        for layer in base_model.layers[:-trainable_layers]:
            layer.trainable = False

        print(f"✓ Unfrozen last {trainable_layers} layers")
        print(f"  Trainable parameters: {sum([K.count_params(w) for w in self.model.trainable_weights]):,}")

        # Recompile with lower learning rate
        self.compile_model(learning_rate=self.config['INITIAL_LEARNING_RATE'] / 10)
        callbacks = self.create_callbacks('stage2')

        history = self.model.fit(
            self.train_datagen.flow(
                self.X_train, self.y_train_cat,
                batch_size=self.config['BATCH_SIZE']
            ),
            epochs=self.config['STAGE2_EPOCHS'],
            validation_data=(self.X_val, self.y_val_cat),
            class_weight=self.class_weights,
            callbacks=callbacks,
            verbose=1
        )

        return history

    def train(self):
        """Main training function"""
        if self.config['USE_TWO_STAGE']:
            history1 = self.train_stage1()
            history2 = self.train_stage2()

            # Combine histories
            self.history = {
                'loss': history1.history['loss'] + history2.history['loss'],
                'accuracy': history1.history['accuracy'] + history2.history['accuracy'],
                'val_loss': history1.history['val_loss'] + history2.history['val_loss'],
                'val_accuracy': history1.history['val_accuracy'] + history2.history['val_accuracy'],
            }
        else:
            self.compile_model()
            callbacks = self.create_callbacks('single_stage')

            history = self.model.fit(
                self.train_datagen.flow(
                    self.X_train, self.y_train_cat,
                    batch_size=self.config['BATCH_SIZE']
                ),
                epochs=self.config['EPOCHS'],
                validation_data=(self.X_val, self.y_val_cat),
                class_weight=self.class_weights,
                callbacks=callbacks,
                verbose=1
            )

            self.history = history.history

    def evaluate_model(self):
        """Evaluate the model"""
        print("\n" + "="*70)
        print("MODEL EVALUATION")
        print("="*70)

        # Predictions
        y_pred_train = self.model.predict(self.X_train, verbose=0)
        y_pred_test = self.model.predict(self.X_test, verbose=0)
        y_pred_val = self.model.predict(self.X_val, verbose=0)

        # Convert to class labels
        y_pred_train_classes = np.argmax(y_pred_train, axis=1)
        y_pred_test_classes = np.argmax(y_pred_test, axis=1)
        y_pred_val_classes = np.argmax(y_pred_val, axis=1)

        # Calculate metrics
        from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

        results = []
        for name, y_true, y_pred in [
            ('Train', self.y_train, y_pred_train_classes),
            ('Test', self.y_test, y_pred_test_classes),
            ('Val', self.y_val, y_pred_val_classes)
        ]:
            acc = accuracy_score(y_true, y_pred)
            f1 = f1_score(y_true, y_pred, average='weighted')
            prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
            rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)

            results.append({
                'Dataset': name,
                'Accuracy': acc,
                'F1-Score': f1,
                'Precision': prec,
                'Recall': rec
            })

            print(f"\n{name} Set:")
            print(f"  Accuracy:  {acc:.4f}")
            print(f"  F1-Score:  {f1:.4f}")
            print(f"  Precision: {prec:.4f}")
            print(f"  Recall:    {rec:.4f}")

        self.results_df = pd.DataFrame(results)

        # Classification report
        print("\n" + "-"*70)
        print("CLASSIFICATION REPORT (Test Set)")
        print("-"*70)
        class_names = [self.config['CLASS_NAMES'][i] for i in range(5)]
        print(classification_report(self.y_test, y_pred_test_classes,
                                   target_names=class_names,
                                   zero_division=0))

        # Store predictions
        self.predictions = {
            'train': y_pred_train_classes,
            'test': y_pred_test_classes,
            'val': y_pred_val_classes
        }
        self.probabilities = {
            'test': y_pred_test,
            'val': y_pred_val
        }

    def plot_training_history(self):
        """Plot training history"""
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Accuracy
        axes[0].plot(self.history['accuracy'], label='Train', linewidth=2)
        axes[0].plot(self.history['val_accuracy'], label='Validation', linewidth=2)
        axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Accuracy')
        axes[0].legend()
        axes[0].grid(alpha=0.3)

        # Loss
        axes[1].plot(self.history['loss'], label='Train', linewidth=2)
        axes[1].plot(self.history['val_loss'], label='Validation', linewidth=2)
        axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Loss')
        axes[1].legend()
        axes[1].grid(alpha=0.3)



In [ ]:
import os

extract_dir = "/content/extracted_files/diabeties"
print(os.listdir(extract_dir))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import os
from glob import glob
from tqdm import tqdm
import zipfile
import warnings
warnings.filterwarnings('ignore')

# TensorFlow/Keras imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import EfficientNetB3, ResNet50V2, DenseNet121
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    Dense, Dropout, GlobalAveragePooling2D, BatchNormalization,
    Conv2D, MaxPooling2D, Flatten, Input, Concatenate, Add
)
from tensorflow.keras.optimizers import Adam, SGD, RMSprop
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau,
    LearningRateScheduler, TensorBoard, CSVLogger
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.regularizers import l2
from tensorflow.keras import backend as K

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.utils.class_weight import compute_class_weight
import pickle

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# GPU Configuration
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    for device in physical_devices:
        tf.config.experimental.set_memory_growth(device, True)
    print(f"✓ GPU Available: {len(physical_devices)} device(s)")
else:
    print("⚠ No GPU found, using CPU")

# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
except:
    print("Not running in Colab")

# Configuration
CONFIG = {
    # Data
    'IMG_SIZE': (224, 224),
    'DATA_DIR': '/content/diabeties',
    'ZIP_PATH': '/content/drive/MyDrive/diabeties.zip',
    'BATCH_SIZE': 32,
    'NUM_CLASSES': 5,
    'RANDOM_STATE': 42,

    # Model Architecture
    'BASE_MODEL': 'EfficientNetB3',  # Options: EfficientNetB3, ResNet50V2, DenseNet121, Custom
    'TRAINABLE_LAYERS': 50,  # Number of layers to unfreeze for fine-tuning
    'DROPOUT_RATE': 0.3,
    'L2_REGULARIZATION': 0.0001,

    # Training Strategy
    'EPOCHS': 100,
    'INITIAL_LEARNING_RATE': 0.001,
    'USE_MIXED_PRECISION': True,  # Faster training on modern GPUs

    # Learning Rate Schedule
    'LR_SCHEDULE': 'cosine_decay_restart',  # Options: reduce_on_plateau, cosine_decay, cosine_decay_restart, exponential, step_decay, custom

    # Callbacks
    'EARLY_STOPPING_PATIENCE': 15,
    'REDUCE_LR_PATIENCE': 5,
    'REDUCE_LR_FACTOR': 0.5,
    'MIN_LR': 1e-7,

    # Data Augmentation
    'USE_AUGMENTATION': True,
    'AUGMENTATION_STRENGTH': 'medium',  # light, medium, strong

    # Advanced Training Techniques
    'USE_CLASS_WEIGHTS': True,
    'USE_FOCAL_LOSS': True,  # Better for imbalanced data
    'USE_LABEL_SMOOTHING': 0.1,  # Prevents overconfidence
    'USE_MIXUP': True,  # Data augmentation in feature space
    'MIXUP_ALPHA': 0.2,
    'USE_GRADIENT_CLIPPING': True,
    'GRADIENT_CLIP_VALUE': 1.0,

    # Two-Stage Training
    'USE_TWO_STAGE': True,
    'STAGE1_EPOCHS': 20,  # Train only top layers
    'STAGE2_EPOCHS': 80,   # Fine-tune entire model

    'CLASS_NAMES': {
        0: 'No DR',
        1: 'Mild',
        2: 'Moderate',
        3: 'Severe',
        4: 'Proliferative DR'
    }
}

# Enable mixed precision for faster training
if CONFIG['USE_MIXED_PRECISION']:
    policy = tf.keras.mixed_precision.Policy('mixed_float16')
    tf.keras.mixed_precision.set_global_policy(policy)
    print("✓ Mixed precision enabled")


# Custom Learning Rate Schedulers
class CustomLRScheduler:
    """Collection of learning rate schedules"""

    @staticmethod
    def cosine_decay_with_warmup(epoch, total_epochs, initial_lr, warmup_epochs=5):
        """Cosine decay with warmup"""
        if epoch < warmup_epochs:
            return initial_lr * (epoch + 1) / warmup_epochs

        progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
        return initial_lr * 0.5 * (1 + np.cos(np.pi * progress))

    @staticmethod
    def cosine_decay_with_restart(epoch, initial_lr, restart_epochs=20):
        """Cosine annealing with warm restarts"""
        cycle = np.floor(1 + epoch / restart_epochs)
        x = np.abs(epoch / restart_epochs - cycle)
        return initial_lr * 0.5 * (1 + np.cos(np.pi * x))

    @staticmethod
    def exponential_decay(epoch, initial_lr, decay_rate=0.95):
        """Exponential decay"""
        return initial_lr * (decay_rate ** epoch)

    @staticmethod
    def step_decay(epoch, initial_lr, drop_rate=0.5, epochs_drop=10):
        """Step decay"""
        return initial_lr * (drop_rate ** np.floor(epoch / epochs_drop))


# Custom Losses
class FocalLoss(tf.keras.losses.Loss):
    """Focal Loss for handling class imbalance"""

    def __init__(self, alpha=0.25, gamma=2.0, from_logits=False):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.from_logits = from_logits

    def call(self, y_true, y_pred):
        if self.from_logits:
            y_pred = tf.nn.softmax(y_pred, axis=-1)

        # Clip predictions to prevent log(0)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)

        # Calculate cross entropy
        ce = -y_true * tf.math.log(y_pred)

        # Calculate focal loss
        weight = self.alpha * y_true * tf.pow(1 - y_pred, self.gamma)
        focal_loss = weight * ce

        return tf.reduce_sum(focal_loss, axis=-1)


def mixup_data(x, y, alpha=0.2):
    """Apply mixup augmentation"""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = x.shape[0]
    index = np.random.permutation(batch_size)

    mixed_x = lam * x + (1 - lam) * x[index]
    mixed_y = lam * y + (1 - lam) * y[index]

    return mixed_x, mixed_y


class DeepLearningDRClassifier:
    """Advanced Deep Learning Classifier for Diabetic Retinopathy"""

    def __init__(self, config):
        self.config = config
        self.model = None
        self.history = None
        self.class_weights = None

    def extract_dataset(self):
        """Extract zip file"""
        if os.path.exists(self.config['DATA_DIR']):
            print(f"✓ Dataset already extracted")
            return

        print("Extracting dataset...")
        with zipfile.ZipFile(self.config['ZIP_PATH'], 'r') as zip_ref:
            zip_ref.extractall('/content/')
        print("✓ Extraction complete")

    def load_images_from_split(self, split):
        """Load images and labels"""
        images = []
        labels = []

        split_path = os.path.join(self.config['DATA_DIR'], split)
        if not os.path.exists(split_path):
            return np.array(images), np.array(labels)

        subdirs = sorted([d for d in os.listdir(split_path)
                         if os.path.isdir(os.path.join(split_path, d))])

        for subdir in subdirs:
            class_folder = os.path.join(split_path, subdir)
            class_num = None
            for i in range(5):
                if f'class{i}' in subdir.lower():
                    class_num = i
                    break

            if class_num is None:
                continue

            extensions = ['*.jpg', '*.png', '*.jpeg', '*.JPG', '*.PNG', '*.JPEG']
            image_files = []
            for ext in extensions:
                image_files.extend(glob(os.path.join(class_folder, ext)))

            print(f"  Loading {len(image_files)} images from {split}/{subdir} (Class {class_num})")

            for img_path in tqdm(image_files, desc=f"{split}-{subdir}", leave=False):
                try:
                    img = cv2.imread(img_path)
                    if img is None:
                        continue

                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    img = self._preprocess_image(img)
                    img = cv2.resize(img, self.config['IMG_SIZE'])

                    images.append(img)
                    labels.append(class_num)

                except Exception as e:
                    continue

        return np.array(images), np.array(labels)

    def _preprocess_image(self, img):
        """Advanced preprocessing for retinal images"""
        # Apply CLAHE
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        l = clahe.apply(l)
        img = cv2.merge([l, a, b])
        img = cv2.cvtColor(img, cv2.COLOR_LAB2RGB)

        # Gaussian blur
        img = cv2.GaussianBlur(img, (3, 3), 0)

        return img

    def load_data(self):
        """Load all datasets"""
        print("\n" + "="*70)
        print("LOADING DATASET")
        print("="*70)

        self.X_train, self.y_train = self.load_images_from_split('train')
        self.X_test, self.y_test = self.load_images_from_split('test')
        self.X_val, self.y_val = self.load_images_from_split('val')

        # Normalize images
        self.X_train = self.X_train.astype('float32') / 255.0
        self.X_test = self.X_test.astype('float32') / 255.0
        self.X_val = self.X_val.astype('float32') / 255.0

        # Convert labels to categorical
        self.y_train_cat = tf.keras.utils.to_categorical(self.y_train, self.config['NUM_CLASSES'])
        self.y_test_cat = tf.keras.utils.to_categorical(self.y_test, self.config['NUM_CLASSES'])
        self.y_val_cat = tf.keras.utils.to_categorical(self.y_val, self.config['NUM_CLASSES'])

        print(f"\n✓ Dataset loaded:")
        print(f"  Train: {self.X_train.shape}")
        print(f"  Test:  {self.X_test.shape}")
        print(f"  Val:   {self.X_val.shape}")

        self._print_class_distribution()
        self._calculate_class_weights()

    def _print_class_distribution(self):
        """Print class distribution"""
        print("\n" + "="*70)
        print("CLASS DISTRIBUTION")
        print("="*70)

        for name, labels in [('Train', self.y_train), ('Test', self.y_test), ('Val', self.y_val)]:
            counts = np.bincount(labels, minlength=5)
            total = len(labels)
            print(f"\n{name} ({total} images):")
            for cls, count in enumerate(counts):
                cls_name = self.config['CLASS_NAMES'][cls]
                percentage = (count / total * 100) if total > 0 else 0
                print(f"  {cls_name:20s}: {count:5d} ({percentage:5.1f}%)")

    def _calculate_class_weights(self):
        """Calculate class weights for imbalanced data"""
        if self.config['USE_CLASS_WEIGHTS']:
            classes = np.unique(self.y_train)
            weights = compute_class_weight('balanced', classes=classes, y=self.y_train)
            self.class_weights = dict(zip(classes, weights))

            print("\n" + "="*70)
            print("CLASS WEIGHTS")
            print("="*70)
            for cls, weight in self.class_weights.items():
                print(f"  {self.config['CLASS_NAMES'][cls]:20s}: {weight:.3f}")

    def create_data_generators(self):
        """Create data generators with augmentation"""
        print("\n" + "="*70)
        print("DATA AUGMENTATION")
        print("="*70)

        if self.config['USE_AUGMENTATION']:
            strength = self.config['AUGMENTATION_STRENGTH']

            if strength == 'light':
                aug_params = {
                    'rotation_range': 10,
                    'width_shift_range': 0.1,
                    'height_shift_range': 0.1,
                    'zoom_range': 0.1,
                    'horizontal_flip': True,
                    'fill_mode': 'reflect'
                }
            elif strength == 'medium':
                aug_params = {
                    'rotation_range': 20,
                    'width_shift_range': 0.15,
                    'height_shift_range': 0.15,
                    'shear_range': 0.15,
                    'zoom_range': 0.15,
                    'horizontal_flip': True,
                    'vertical_flip': True,
                    'brightness_range': [0.8, 1.2],
                    'fill_mode': 'reflect'
                }
            else:  # strong
                aug_params = {
                    'rotation_range': 30,
                    'width_shift_range': 0.2,
                    'height_shift_range': 0.2,
                    'shear_range': 0.2,
                    'zoom_range': 0.2,
                    'horizontal_flip': True,
                    'vertical_flip': True,
                    'brightness_range': [0.7, 1.3],
                    'fill_mode': 'reflect'
                }

            print(f"✓ Using {strength} augmentation")
            self.train_datagen = ImageDataGenerator(**aug_params)
        else:
            print("✓ No augmentation")
            self.train_datagen = ImageDataGenerator()

        self.val_datagen = ImageDataGenerator()

    def build_model(self):
        """Build the deep learning model"""
        print("\n" + "="*70)
        print("BUILDING MODEL")
        print("="*70)

        input_shape = (*self.config['IMG_SIZE'], 3)

        if self.config['BASE_MODEL'] == 'EfficientNetB3':
            base_model = EfficientNetB3(
                include_top=False,
                weights='imagenet',
                input_shape=input_shape
            )
        elif self.config['BASE_MODEL'] == 'ResNet50V2':
            base_model = ResNet50V2(
                include_top=False,
                weights='imagenet',
                input_shape=input_shape
            )
        elif self.config['BASE_MODEL'] == 'DenseNet121':
            base_model = DenseNet121(
                include_top=False,
                weights='imagenet',
                input_shape=input_shape
            )
        else:
            raise ValueError(f"Unknown base model: {self.config['BASE_MODEL']}")

        # Freeze base model initially
        base_model.trainable = False

        # Build top layers
        inputs = Input(shape=input_shape)
        x = base_model(inputs, training=False)
        x = GlobalAveragePooling2D()(x)
        x = BatchNormalization()(x)
        x = Dropout(self.config['DROPOUT_RATE'])(x)
        x = Dense(512, activation='relu', kernel_regularizer=l2(self.config['L2_REGULARIZATION']))(x)
        x = BatchNormalization()(x)
        x = Dropout(self.config['DROPOUT_RATE'])(x)
        x = Dense(256, activation='relu', kernel_regularizer=l2(self.config['L2_REGULARIZATION']))(x)
        x = Dropout(self.config['DROPOUT_RATE'] / 2)(x)

        # Output layer
        if self.config['USE_MIXED_PRECISION']:
            outputs = Dense(self.config['NUM_CLASSES'], activation='softmax', dtype='float32')(x)
        else:
            outputs = Dense(self.config['NUM_CLASSES'], activation='softmax')(x)

        self.model = Model(inputs, outputs)

        print(f"✓ Model built: {self.config['BASE_MODEL']}")
        print(f"  Total parameters: {self.model.count_params():,}")
        print(f"  Trainable parameters: {sum([K.count_params(w) for w in self.model.trainable_weights]):,}")

    def compile_model(self, learning_rate=None):
        """Compile the model"""
        if learning_rate is None:
            learning_rate = self.config['INITIAL_LEARNING_RATE']

        # Optimizer with gradient clipping
        if self.config['USE_GRADIENT_CLIPPING']:
            optimizer = Adam(
                learning_rate=learning_rate,
                clipvalue=self.config['GRADIENT_CLIP_VALUE']
            )
        else:
            optimizer = Adam(learning_rate=learning_rate)

        # Loss function
        if self.config['USE_FOCAL_LOSS']:
            loss = FocalLoss(alpha=0.25, gamma=2.0)
            print("✓ Using Focal Loss")
        elif self.config['USE_LABEL_SMOOTHING'] > 0:
            loss = tf.keras.losses.CategoricalCrossentropy(
                label_smoothing=self.config['USE_LABEL_SMOOTHING']
            )
            print(f"✓ Using Label Smoothing: {self.config['USE_LABEL_SMOOTHING']}")
        else:
            loss = 'categorical_crossentropy'

        self.model.compile(
            optimizer=optimizer,
            loss=loss,
            metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
        )

    def create_callbacks(self, stage='stage1'):
        """Create training callbacks"""
        callbacks = []

        # Model checkpoint
        checkpoint = ModelCheckpoint(
            f'best_model_{stage}.h5',
            monitor='val_accuracy',
            save_best_only=True,
            mode='max',
            verbose=1
        )
        callbacks.append(checkpoint)

        # Early stopping
        early_stop = EarlyStopping(
            monitor='val_loss',
            patience=self.config['EARLY_STOPPING_PATIENCE'],
            restore_best_weights=True,
            verbose=1
        )
        callbacks.append(early_stop)

        # Learning rate schedule
        lr_schedule = self.config['LR_SCHEDULE']

        if lr_schedule == 'reduce_on_plateau':
            reduce_lr = ReduceLROnPlateau(
                monitor='val_loss',
                factor=self.config['REDUCE_LR_FACTOR'],
                patience=self.config['REDUCE_LR_PATIENCE'],
                min_lr=self.config['MIN_LR'],
                verbose=1
            )
            callbacks.append(reduce_lr)
            print(f"✓ Using ReduceLROnPlateau")

        elif lr_schedule == 'cosine_decay':
            lr_scheduler = LearningRateScheduler(
                lambda epoch: CustomLRScheduler.cosine_decay_with_warmup(
                    epoch,
                    self.config['EPOCHS'],
                    self.config['INITIAL_LEARNING_RATE']
                )
            )
            callbacks.append(lr_scheduler)
            print(f"✓ Using Cosine Decay with Warmup")

        elif lr_schedule == 'cosine_decay_restart':
            lr_scheduler = LearningRateScheduler(
                lambda epoch: CustomLRScheduler.cosine_decay_with_restart(
                    epoch,
                    self.config['INITIAL_LEARNING_RATE'],
                    restart_epochs=20
                )
            )
            callbacks.append(lr_scheduler)
            print(f"✓ Using Cosine Decay with Restarts")

        elif lr_schedule == 'exponential':
            lr_scheduler = LearningRateScheduler(
                lambda epoch: CustomLRScheduler.exponential_decay(
                    epoch,
                    self.config['INITIAL_LEARNING_RATE']
                )
            )
            callbacks.append(lr_scheduler)
            print(f"✓ Using Exponential Decay")

        elif lr_schedule == 'step_decay':
            lr_scheduler = LearningRateScheduler(
                lambda epoch: CustomLRScheduler.step_decay(
                    epoch,
                    self.config['INITIAL_LEARNING_RATE']
                )
            )
            callbacks.append(lr_scheduler)
            print(f"✓ Using Step Decay")

        # CSV Logger
        csv_logger = CSVLogger(f'training_log_{stage}.csv')
        callbacks.append(csv_logger)

        # TensorBoard
        tensorboard = TensorBoard(
            log_dir=f'./logs/{stage}',
            histogram_freq=0,
            write_graph=False
        )
        callbacks.append(tensorboard)

        return callbacks

    def train_stage1(self):
        """Stage 1: Train only top layers"""
        print("\n" + "="*70)
        print("STAGE 1: TRAINING TOP LAYERS")
        print("="*70)

        self.compile_model()
        callbacks = self.create_callbacks('stage1')

        history = self.model.fit(
            self.train_datagen.flow(
                self.X_train, self.y_train_cat,
                batch_size=self.config['BATCH_SIZE']
            ),
            epochs=self.config['STAGE1_EPOCHS'],
            validation_data=(self.X_val, self.y_val_cat),
            class_weight=self.class_weights,
            callbacks=callbacks,
            verbose=1
        )

        return history

    def train_stage2(self):
        """Stage 2: Fine-tune entire model"""
        print("\n" + "="*70)
        print("STAGE 2: FINE-TUNING ENTIRE MODEL")
        print("="*70)

        # Unfreeze base model layers
        base_model = self.model.layers[1]
        base_model.trainable = True

        # Freeze early layers, train later layers
        trainable_layers = self.config['TRAINABLE_LAYERS']
        for layer in base_model.layers[:-trainable_layers]:
            layer.trainable = False

        print(f"✓ Unfrozen last {trainable_layers} layers")
        print(f"  Trainable parameters: {sum([K.count_params(w) for w in self.model.trainable_weights]):,}")

        # Recompile with lower learning rate
        self.compile_model(learning_rate=self.config['INITIAL_LEARNING_RATE'] / 10)
        callbacks = self.create_callbacks('stage2')

        history = self.model.fit(
            self.train_datagen.flow(
                self.X_train, self.y_train_cat,
                batch_size=self.config['BATCH_SIZE']
            ),
            epochs=self.config['STAGE2_EPOCHS'],
            validation_data=(self.X_val, self.y_val_cat),
            class_weight=self.class_weights,
            callbacks=callbacks,
            verbose=1
        )

        return history

    def train(self):
        """Main training function"""
        if self.config['USE_TWO_STAGE']:
            history1 = self.train_stage1()
            history2 = self.train_stage2()

            # Combine histories
            self.history = {
                'loss': history1.history['loss'] + history2.history['loss'],
                'accuracy': history1.history['accuracy'] + history2.history['accuracy'],
                'val_loss': history1.history['val_loss'] + history2.history['val_loss'],
                'val_accuracy': history1.history['val_accuracy'] + history2.history['val_accuracy'],
            }
        else:
            self.compile_model()
            callbacks = self.create_callbacks('single_stage')

            history = self.model.fit(
                self.train_datagen.flow(
                    self.X_train, self.y_train_cat,
                    batch_size=self.config['BATCH_SIZE']
                ),
                epochs=self.config['EPOCHS'],
                validation_data=(self.X_val, self.y_val_cat),
                class_weight=self.class_weights,
                callbacks=callbacks,
                verbose=1
            )

            self.history = history.history

    def evaluate_model(self):
        """Evaluate the model"""
        print("\n" + "="*70)
        print("MODEL EVALUATION")
        print("="*70)

        # Predictions
        y_pred_train = self.model.predict(self.X_train, verbose=0)
        y_pred_test = self.model.predict(self.X_test, verbose=0)
        y_pred_val = self.model.predict(self.X_val, verbose=0)

        # Convert to class labels
        y_pred_train_classes = np.argmax(y_pred_train, axis=1)
        y_pred_test_classes = np.argmax(y_pred_test, axis=1)
        y_pred_val_classes = np.argmax(y_pred_val, axis=1)

        # Calculate metrics
        from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

        results = []
        for name, y_true, y_pred in [
            ('Train', self.y_train, y_pred_train_classes),
            ('Test', self.y_test, y_pred_test_classes),
            ('Val', self.y_val, y_pred_val_classes)
        ]:
            acc = accuracy_score(y_true, y_pred)
            f1 = f1_score(y_true, y_pred, average='weighted')
            prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
            rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)

            results.append({
                'Dataset': name,
                'Accuracy': acc,
                'F1-Score': f1,
                'Precision': prec,
                'Recall': rec
            })

            print(f"\n{name} Set:")
            print(f"  Accuracy:  {acc:.4f}")
            print(f"  F1-Score:  {f1:.4f}")
            print(f"  Precision: {prec:.4f}")
            print(f"  Recall:    {rec:.4f}")

        self.results_df = pd.DataFrame(results)

        # Classification report
        print("\n" + "-"*70)
        print("CLASSIFICATION REPORT (Test Set)")
        print("-"*70)
        class_names = [self.config['CLASS_NAMES'][i] for i in range(5)]
        print(classification_report(self.y_test, y_pred_test_classes,
                                   target_names=class_names,
                                   zero_division=0))

        # Store predictions
        self.predictions = {
            'train': y_pred_train_classes,
            'test': y_pred_test_classes,
            'val': y_pred_val_classes
        }
        self.probabilities = {
            'test': y_pred_test,
            'val': y_pred_val
        }

    def plot_training_history(self):
        """Plot training history"""
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Accuracy
        axes[0].plot(self.history['accuracy'], label='Train', linewidth=2)
        axes[0].plot(self.history['val_accuracy'], label='Validation', linewidth=2)
        axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Accuracy')
        axes[0].legend()
        axes[0].grid(alpha=0.3)

        # Loss
        axes[1].plot(self.history['loss'], label='Train', linewidth=2)
        axes[1].plot(self.history['val_loss'], label='Validation', linewidth=2)
        axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Loss')
        axes[1].legend()
        axes[1].grid(alpha=0.3)

        plt.tight_layout()
        plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
        plt.show()

    def visualize_results(self):
        """Create comprehensive visualizations"""
        # Confusion matrices
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        datasets = [
            ('Train', self.y_train, self.predictions['train']),
            ('Test', self.y_test, self.predictions['test']),
            ('Validation', self.y_val, self.predictions['val'])
        ]

        for idx, (name, y_true, y_pred) in enumerate(datasets):
            cm = confusion_matrix(y_true, y_pred)
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], cbar=True)
            axes[idx].set_title(f'{name} Set Confusion Matrix', fontsize=12, fontweight='bold')
            axes[idx].set_xlabel('Predicted Class')
            axes[idx].set_ylabel('Actual Class')

        plt.tight_layout()
        plt.savefig('dl_confusion_matrices.png', dpi=300, bbox_inches='tight')
        plt.show()

        # ROC curves
        self._plot_roc_curves()

        # Metrics comparison
        self._plot_metrics_comparison()

        # Results summary
        print("\n" + "="*70)
        print("RESULTS SUMMARY")
        print("="*70)
        print(self.results_df.to_string(index=False))

    def _plot_roc_curves(self):
        """Plot ROC curves"""
        from sklearn.preprocessing import label_binarize

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        for idx, (name, y_true, y_proba) in enumerate([
            ('Test', self.y_test, self.probabilities['test']),
            ('Validation', self.y_val, self.probabilities['val'])
        ]):
            y_true_bin = label_binarize(y_true, classes=range(5))

            for i in range(5):
                fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_proba[:, i])
                auc_score = roc_auc_score(y_true_bin[:, i], y_proba[:, i])
                axes[idx].plot(fpr, tpr, lw=2,
                             label=f'{self.config["CLASS_NAMES"][i]} (AUC={auc_score:.3f})')

            axes[idx].plot([0, 1], [0, 1], 'k--', lw=2, label='Random')
            axes[idx].set_xlabel('False Positive Rate', fontsize=11)
            axes[idx].set_ylabel('True Positive Rate', fontsize=11)
            axes[idx].set_title(f'ROC Curves - {name} Set', fontsize=12, fontweight='bold')
            axes[idx].legend(loc='lower right', fontsize=9)
            axes[idx].grid(alpha=0.3)

        plt.tight_layout()
        plt.savefig('dl_roc_curves.png', dpi=300, bbox_inches='tight')
        plt.show()

    def _plot_metrics_comparison(self):
        """Plot metrics comparison"""
        fig, ax = plt.subplots(1, 1, figsize=(10, 6))

        x = np.arange(len(self.results_df))
        width = 0.2

        metrics = ['Accuracy', 'F1-Score', 'Precision', 'Recall']
        colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']

        for i, (metric, color) in enumerate(zip(metrics, colors)):
            ax.bar(x + i*width, self.results_df[metric], width,
                   label=metric, color=color, alpha=0.8)

        ax.set_xlabel('Dataset', fontsize=12)
        ax.set_ylabel('Score', fontsize=12)
        ax.set_title('Deep Learning Model Performance', fontsize=14, fontweight='bold')
        ax.set_xticks(x + width * 1.5)
        ax.set_xticklabels(self.results_df['Dataset'])
        ax.legend()
        ax.set_ylim([0, 1.1])
        ax.grid(axis='y', alpha=0.3)

        plt.tight_layout()
        plt.savefig('dl_metrics_comparison.png', dpi=300, bbox_inches='tight')
        plt.show()

    def save_model(self, output_dir='dl_model'):
        """Save the model and configuration"""
        os.makedirs(output_dir, exist_ok=True)

        # Save model
        self.model.save(f'{output_dir}/model.h5')
        print(f"✓ Model saved to {output_dir}/model.h5")

        # Save model in SavedModel format (for deployment)
        self.model.save(f'{output_dir}/saved_model')
        print(f"✓ Model saved to {output_dir}/saved_model/")

        # Save weights only
        self.model.save_weights(f'{output_dir}/model_weights.h5')

        # Save training history
        with open(f'{output_dir}/history.pkl', 'wb') as f:
            pickle.dump(self.history, f)

        # Save configuration
        with open(f'{output_dir}/config.pkl', 'wb') as f:
            pickle.dump(self.config, f)

        # Save results
        self.results_df.to_csv(f'{output_dir}/results.csv', index=False)

        print(f"✓ All files saved to {output_dir}/")

    def predict_image(self, image_path):
        """Predict class for a new image"""
        img = cv2.imread(image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = self._preprocess_image(img)
        img = cv2.resize(img, self.config['IMG_SIZE'])
        img = img.astype('float32') / 255.0
        img = np.expand_dims(img, axis=0)

        # Predict
        prediction = self.model.predict(img, verbose=0)[0]
        class_id = np.argmax(prediction)
        confidence = prediction[class_id]
        severity = self.config['CLASS_NAMES'][class_id]

        return class_id, severity, confidence, prediction


# Main execution
if __name__ == "__main__":
    print("="*70)
    print("ADVANCED DEEP LEARNING DIABETIC RETINOPATHY CLASSIFIER")
    print("="*70)
    print("\nConfiguration:")
    print(f"  Base Model: {CONFIG['BASE_MODEL']}")
    print(f"  Epochs: {CONFIG['EPOCHS']}")
    print(f"  Learning Rate: {CONFIG['INITIAL_LEARNING_RATE']}")
    print(f"  LR Schedule: {CONFIG['LR_SCHEDULE']}")
    print(f"  Batch Size: {CONFIG['BATCH_SIZE']}")
    print(f"  Augmentation: {CONFIG['AUGMENTATION_STRENGTH']}")
    print(f"  Two-Stage Training: {CONFIG['USE_TWO_STAGE']}")
    print(f"  Focal Loss: {CONFIG['USE_FOCAL_LOSS']}")
    print(f"  Label Smoothing: {CONFIG['USE_LABEL_SMOOTHING']}")
    print(f"  Mixed Precision: {CONFIG['USE_MIXED_PRECISION']}")

    # Initialize classifier
    classifier = DeepLearningDRClassifier(CONFIG)

    # Extract dataset
    classifier.extract_dataset()

    # Load data
    classifier.load_data()

    # Create data generators
    classifier.create_data_generators()

    # Build model
    classifier.build_model()

    # Display model summary
    print("\n" + "="*70)
    print("MODEL ARCHITECTURE")
    print("="*70)
    classifier.model.summary()

    # Train model
    print("\n" + "="*70)
    print("STARTING TRAINING")
    print("="*70)
    classifier.train()

    # Plot training history
    classifier.plot_training_history()

    # Evaluate model
    classifier.evaluate_model()

    # Visualize results
    classifier.visualize_results()

    # Save model
    classifier.save_model()

    print("\n" + "="*70)
    print("TRAINING COMPLETE!")
    print("="*70)
    print("\nBest results:")
    print(classifier.results_df.to_string(index=False))

    # Example prediction
    # class_id, severity, confidence, proba = classifier.predict_image('test_image.jpg')
    # print(f"\nPrediction: {severity} (Class {class_id})")
    # print(f"Confidence: {confidence:.2%}")
    # print(f"\nAll class probabilities:")
    # for i, prob in enumerate(proba):
    #     print(f"  {CONFIG['CLASS_NAMES'][i]:20s}: {prob:.4f}")

    # Load saved model for inference (example)
    # loaded_model = tf.keras.models.load_model('dl_model/model.h5',
    #                                           custom_objects={'FocalLoss': FocalLoss})

In [ ]:
# %% [code]
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, cohen_kappa_score
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# -----------------------------
# 1️⃣ Metrics function
# -----------------------------
def print_metrics(y_true, y_pred, name):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')
    kappa = cohen_kappa_score(y_true, y_pred)
    print(f"{name}: Accuracy={acc:.3f}, F1-macro={f1:.3f}, Cohen-Kappa={kappa:.3f}")
    return acc, f1, kappa

# -----------------------------
# 2️⃣ Confusion matrix plot
# -----------------------------
def plot_cm(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    cmn = cm / (cm.sum(axis=1, keepdims=True) + 1e-9)
    plt.figure(figsize=(5,4))
    sns.heatmap(cmn, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=train_ds.classes, yticklabels=train_ds.classes)
    plt.title(title)
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.tight_layout()
    plt.show()

# -----------------------------
# 3️⃣ Classical predictions
# -----------------------------
# Ensure predictions cover full test set
# SVM
svm_pred = svm.predict(Xte_pca)
# MLP
mlp_pred = mlp.predict(X_test)  # assuming you have mlp trained
# Pick best classical model
if f1_score(y_test, mlp_pred, average='macro') >= f1_score(y_test, svm_pred, average='macro'):
    best_classical_pred = mlp_pred
    best_classical_name = "CNN + MLP"
else:
    best_classical_pred = svm_pred
    best_classical_name = "CNN + SVM"

# -----------------------------
# 4️⃣ QSVM predictions (subset)
# -----------------------------
# Only first len(qs_pred) of y_test
qs_pred_corrected = qs_pred
y_test_qs = y_test[:len(qs_pred_corrected)]

# -----------------------------
# 5️⃣ Print metrics
# -----------------------------
print_metrics(y_test, svm_pred, "CNN + SVM")
print_metrics(y_test, mlp_pred, "CNN + MLP")
print_metrics(y_test_qs, qs_pred_corrected, "QCNN + QSVM")

# -----------------------------
# 6️⃣ Plot confusion matrices
# -----------------------------
plot_cm(y_test, best_classical_pred, f"{best_classical_name} — Confusion Matrix")
plot_cm(y_test_qs, qs_pred_corrected, "QCNN + QSVM — Confusion Matrix")


In [ ]:
# 🚀 Quantum–Classical Hybrid DR Classification Pipeline
# ===============================================
# Full Colab-ready version — works with `/content/drive/MyDrive/diabeties.zip`
# Structure inside zip should be:
# diabeties/train/0..4 , diabeties/test/0..4 , diabeties/eval/0..4

# ----------------------------------------
# 1️⃣ SETUP & INSTALL
# ----------------------------------------
!pip install timm pennylane scikit-learn torch torchvision torchaudio matplotlib tqdm

import os, zipfile, torch, timm, numpy as np
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import f1_score, accuracy_score, cohen_kappa_score
from tqdm import tqdm
import pennylane as qml
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight

# ----------------------------------------
# 2️⃣ MOUNT & EXTRACT DATA
# ----------------------------------------
from google.colab import drive
drive.mount('/content/drive')

zip_path = "/content/drive/MyDrive/diabeties.zip"
extract_dir = "/content/diabeties"

if not os.path.exists(extract_dir):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall("/content/")
print("✅ Data extracted to:", extract_dir)

# ----------------------------------------
# 3️⃣ DATA LOADERS
# ----------------------------------------
BATCH = 16
IMG_SIZE = 224
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

train_ds = datasets.ImageFolder(os.path.join(extract_dir, 'train'), transform)
test_ds  = datasets.ImageFolder(os.path.join(extract_dir, 'test'), transform)
eval_ds  = datasets.ImageFolder(os.path.join(extract_dir, 'val'), transform)

train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False)
eval_dl  = DataLoader(eval_ds,  batch_size=BATCH, shuffle=False)

NUM_CLASSES = len(train_ds.classes)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ----------------------------------------
# 4️⃣ EFFICIENTNET FEATURE EXTRACTOR
# ----------------------------------------
class EffNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b3a', pretrained=True)
        self.backbone.classifier = nn.Linear(self.backbone.classifier.in_features, num_classes)

    def forward(self, x):
        return self.backbone(x)

cnn_model = EffNet(NUM_CLASSES).to(DEVICE)
opt = AdamW(cnn_model.parameters(), lr=1e-4)
crit = nn.CrossEntropyLoss()

# simple short train loop (for demo; extend epochs for real use)
for epoch in range(1):
    cnn_model.train()
    for xb, yb in tqdm(train_dl, desc=f'Epoch {epoch+1}'):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        loss = crit(cnn_model(xb), yb)
        loss.backward()
        opt.step()

# feature extractor
feature_extractor = nn.Sequential(*(list(cnn_model.backbone.children())[:-1]))
feature_extractor.eval()

def extract_features(loader):
    feats, labels = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            out = feature_extractor(xb).squeeze()
            feats.append(out.cpu().numpy())
            labels.append(yb.numpy())
    return np.vstack(feats), np.hstack(labels)

X_train, y_train = extract_features(train_dl)
X_test,  y_test  = extract_features(test_dl)

# ----------------------------------------
# 5️⃣ SVM BASELINE
# ----------------------------------------
pca = PCA(n_components=256)
Xtr_pca, Xte_pca = pca.fit_transform(X_train), pca.transform(X_test)
svm = SVC(kernel='rbf', class_weight='balanced')
svm.fit(Xtr_pca, y_train)
yp = svm.predict(Xte_pca)
print('SVM acc=%.3f f1=%.3f' % (accuracy_score(y_test, yp), f1_score(y_test, yp, average='macro')))

# ----------------------------------------
# 6️⃣ QUANTUM LAYER HELPERS
# ----------------------------------------
n_qubits = 8
dev = qml.device('default.qubit', wires=n_qubits)

def feature_map(x):
    for i in range(n_qubits):
        qml.RY(x[i], wires=i)
    for i in range(n_qubits-1):
        qml.CNOT(wires=[i, i+1])

def quantum_kernel_matrix(X1, X2):
    mat = np.zeros((len(X1), len(X2)))
    for i, x1 in enumerate(tqdm(X1, desc='Kernel rows')):
        for j, x2 in enumerate(X2):
            def circuit():
                feature_map(x1)
                qml.adjoint(feature_map)(x2)
                return qml.probs(wires=range(n_qubits))
            qnode = qml.QNode(circuit, dev)
            probs = qnode()
            mat[i,j] = probs[0]
    return mat

# ----------------------------------------
# 7️⃣ SIMPLE QSVM
# ----------------------------------------
X8 = PCA(n_components=8).fit_transform(X_train)
X8t = PCA(n_components=8).fit_transform(X_test)

Ktr = quantum_kernel_matrix(X8[:50], X8[:50])  # subsample for speed
Kte = quantum_kernel_matrix(X8t[:20], X8[:50])

qs = SVC(kernel='precomputed', C=5.0)
qs.fit(Ktr, y_train[:50])
yp_qs = qs.predict(Kte)
print('QSVM acc=%.3f f1=%.3f' % (accuracy_score(y_test[:20], yp_qs), f1_score(y_test[:20], yp_qs, average='macro')))

print('\n✅ Pipeline completed successfully!')


In [ ]:
import pennylane as qml
from pennylane import numpy as np

n_qubits = 8
dev = qml.device("default.qubit", wires=n_qubits)

# QCNN encoder circuit
def qcnn_encoder(inputs, weights):
    # Encode input features
    qml.templates.AngleEmbedding(inputs, wires=range(n_qubits))

    # Quantum convolution layer
    for i in range(0, n_qubits, 2):
        qml.CRX(weights[i, 0], wires=[i, (i+1)%n_qubits])
        qml.CRY(weights[i, 1], wires=[i, (i+1)%n_qubits])
        qml.CZ(wires=[i, (i+1)%n_qubits])

    # Pooling layer (measure and reset)
    for i in range(0, n_qubits, 2):
        qml.measure(wires=i)

    # Strongly entangling final layer
    qml.templates.StronglyEntanglingLayers(weights[-1:], wires=range(n_qubits//2))

@qml.qnode(dev)
def qcnn_forward(inputs, weights):
    qcnn_encoder(inputs, weights)
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits//2)]


In [ ]:
# =====================================================
# ⚛️ QCNN Encoder + Classical SVM (Fixed, Colab-Ready)
# =====================================================

import pennylane as qml
from pennylane import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score
import numpy as onp  # use standard numpy for preprocessing

# -----------------------------
# Quantum Device Setup
# -----------------------------
n_qubits = 8
dev = qml.device("default.qubit", wires=n_qubits, shots=None)

# -----------------------------
# QCNN Building Blocks
# -----------------------------
def conv_block(weights, wires):
    """Local rotation + entangling block"""
    for i, w in enumerate(wires):
        qml.RX(weights[i, 0], wires=w)
        qml.RY(weights[i, 1], wires=w)
        qml.RZ(weights[i, 2], wires=w)
    for i in range(len(wires) - 1):
        qml.CNOT(wires=[wires[i], wires[i + 1]])

def pooling_block(wires):
    """Pooling-like layer (unitary compression, not measurement)"""
    for i in range(0, len(wires), 2):
        a, b = wires[i], wires[(i + 1) % len(wires)]
        qml.CRX(0.3, wires=[a, b])
        qml.CRY(0.1, wires=[a, b])
        qml.CNOT(wires=[b, a])

# -----------------------------
# QCNN QNode Definition
# -----------------------------
def make_qcnn_qnode(n_qubits, n_conv_layers=2):
    wires = list(range(n_qubits))

    @qml.qnode(dev, interface="autograd")
    def qnode(inputs, weights):
        # Encode classical inputs
        qml.templates.AngleEmbedding(inputs, wires=wires)

        # Apply convolutional blocks
        for l in range(n_conv_layers):
            conv_block(weights["conv"][l], wires)
            pooling_block(wires)

        # ✅ Fixed: use full tensor for StronglyEntanglingLayers
        qml.templates.StronglyEntanglingLayers(weights["final"], wires=wires)

        # Readout from half the qubits
        readout_wires = wires[: n_qubits // 2]
        return [qml.expval(qml.PauliZ(i)) for i in readout_wires]

    return qnode

# -----------------------------
# Initialize QCNN Weights
# -----------------------------
rng = np.random.default_rng(42)
weights = {
    "conv": rng.normal(0, 0.1, size=(2, n_qubits, 3)),  # 2 conv layers
    "final": rng.normal(0, 0.1, size=(1, n_qubits, 3)), # 1 final block
}

qnode = make_qcnn_qnode(n_qubits=n_qubits, n_conv_layers=2)

# -----------------------------
# Prepare Classical Inputs (8D)
# -----------------------------
pca_q = PCA(n_components=8, random_state=42)
X_train_q = pca_q.fit_transform(X_train)
X_test_q  = pca_q.transform(X_test)

# Normalize to [-π, π] for AngleEmbedding
def normalize_for_angle_embedding(X, mins=None, maxs=None):
    if mins is None:
        mins = X.min(axis=0, keepdims=True)
        maxs = X.max(axis=0, keepdims=True)
    denom = (maxs - mins)
    denom[denom == 0] = 1.0
    Xn = (X - mins) / denom
    return (Xn * 2 * np.pi) - np.pi, mins, maxs

X_train_qn, q_mins, q_maxs = normalize_for_angle_embedding(X_train_q)
X_test_qn, _, _ = normalize_for_angle_embedding(X_test_q, q_mins, q_maxs)

# -----------------------------
# Batch QNode Evaluation
# -----------------------------
def qnode_batch_eval(qnode, X_inputs, weights, batch_size=8):
    q_feats = []
    for i in range(0, len(X_inputs), batch_size):
        batch = X_inputs[i : i + batch_size]
        batch_out = [qnode(x, weights) for x in batch]
        q_feats.append(np.vstack(batch_out))
    return np.vstack(q_feats)

# -----------------------------
# Use full datasets (no subsampling)
# -----------------------------
Xtr_q_sub = X_train_qn    # full training set
ytr_sub = y_train
Xte_q_sub = X_test_qn     # full test set
yte_sub = y_test


print("⚙️ Computing quantum features (may take a while)...")
qtrain_feats = qnode_batch_eval(qnode, Xtr_q_sub, weights, batch_size=8)
qtest_feats = qnode_batch_eval(qnode, Xte_q_sub, weights, batch_size=8)
print("✅ Done. Quantum feature shapes:", qtrain_feats.shape, qtest_feats.shape)

# -----------------------------
# Train SVM on Quantum Features
# -----------------------------
scaler = StandardScaler()
qtr_scaled = scaler.fit_transform(qtrain_feats)
qte_scaled = scaler.transform(qtest_feats)

svm_q = SVC(kernel="rbf", class_weight="balanced")
svm_q.fit(qtr_scaled, ytr_sub)
yp_q = svm_q.predict(qte_scaled)

acc_q = accuracy_score(yte_sub, yp_q)
f1_q = f1_score(yte_sub, yp_q, average="macro")

print(f"\n🎯 QCNN Encoder + SVM Results:")
print(f"Accuracy = {acc_q:.3f}, F1-macro = {f1_q:.3f}")



In [ ]:
# =====================================================
# 🧭 Evaluation: Confusion Matrix + Metrics
# =====================================================
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, cohen_kappa_score

# --- Compute metrics ---
acc_q = accuracy_score(yte_sub, yp_q)
f1_q = f1_score(yte_sub, yp_q, average="macro")
kappa_q = cohen_kappa_score(yte_sub, yp_q)

print("\n🎯 QCNN Encoder + SVM Results on DR Dataset:")
print(f"Accuracy  : {acc_q:.3f}")
print(f"F1-macro : {f1_q:.3f}")
print(f"Cohen’s Kappa : {kappa_q:.3f}")

# --- Confusion Matrix ---
class_names = [str(i) for i in range(NUM_CLASSES)]  # ['0','1','2','3','4']

cm = confusion_matrix(yte_sub, yp_q, labels=list(range(NUM_CLASSES)))
cm_norm = cm / (cm.sum(axis=1, keepdims=True) + 1e-9)

plt.figure(figsize=(6,5))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.title("QCNN + SVM — Confusion Matrix (Normalized)")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

# Optional: print raw counts too
print("\nRaw Confusion Matrix:\n", cm)


In [ ]:
# If your current X_train shape is (N, 1024)
X_train_img = X_train.reshape(-1, 32, 32, 1)  # Example: 1024 = 32*32
X_test_img  = X_test.reshape(-1, 32, 32, 1)


In [ ]:
input_shape = (32, 32, 1)
cnn = Sequential([
    Conv2D(16, (3,3), activation='relu', input_shape=input_shape),
    MaxPooling2D((2,2)),
    Conv2D(32, (3,3), activation='relu'),
    MaxPooling2D((2,2)),
    Flatten(),
    Dense(n_qubits, activation='linear')
])


In [ ]:
X_train_cnn = get_cnn_features(cnn, X_train_img)
X_test_cnn  = get_cnn_features(cnn, X_test_img)


In [ ]:
# Example: 1024 → 32x32 grayscale image
X_train_img = X_train.reshape(-1, 32, 32, 1)
X_test_img  = X_test.reshape(-1, 32, 32, 1)

# CNN input shape
input_shape = (32, 32, 1)
cnn = Sequential([
    Conv2D(16, (3,3), activation='relu', input_shape=input_shape),
    MaxPooling2D((2,2)),
    Conv2D(32, (3,3), activation='relu'),
    MaxPooling2D((2,2)),
    Flatten(),
    Dense(n_qubits, activation='linear')
])




In [ ]:
# =====================================================
# ⚛️ CNN + QCNN Encoder + Classical SVM (Colab-Ready)
# =====================================================

import pennylane as qml
from pennylane import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
import numpy as onp  # classical numpy

# -----------------------------
# Parameters
# -----------------------------
n_qubits = 8
input_shape = (64, 64, 1)  # Example image size (H, W, C)
n_conv_layers = 2  # QCNN conv layers

# -----------------------------
# Classical CNN Feature Extractor
# -----------------------------
cnn = Sequential([
    Conv2D(16, (3,3), activation='relu', input_shape=input_shape),
    MaxPooling2D((2,2)),
    Conv2D(32, (3,3), activation='relu'),
    MaxPooling2D((2,2)),
    Flatten(),
    Dense(n_qubits, activation='linear')  # output 8D features for QCNN
])

# -----------------------------
# Quantum Device
# -----------------------------
dev = qml.device("default.qubit", wires=n_qubits, shots=None)

# -----------------------------
# QCNN Building Blocks
# -----------------------------
def conv_block(weights, wires):
    for i, w in enumerate(wires):
        qml.RX(weights[i,0], wires=w)
        qml.RY(weights[i,1], wires=w)
        qml.RZ(weights[i,2], wires=w)
    for i in range(len(wires)-1):
        qml.CNOT(wires=[wires[i], wires[i+1]])

def pooling_block(wires):
    for i in range(0, len(wires), 2):
        a, b = wires[i], wires[(i+1) % len(wires)]
        qml.CRX(0.3, wires=[a,b])
        qml.CRY(0.1, wires=[a,b])
        qml.CNOT(wires=[b,a])

def make_qcnn_qnode(n_qubits, n_conv_layers=2):
    wires = list(range(n_qubits))
    @qml.qnode(dev, interface="autograd")
    def qnode(inputs, weights):
        qml.templates.AngleEmbedding(inputs, wires=wires)
        for l in range(n_conv_layers):
            conv_block(weights["conv"][l], wires)
            pooling_block(wires)
        qml.templates.StronglyEntanglingLayers(weights["final"], wires=wires)
        return [qml.expval(qml.PauliZ(i)) for i in wires[: n_qubits // 2]]
    return qnode

# -----------------------------
# Initialize QCNN Weights
# -----------------------------
rng = np.random.default_rng(42)
weights = {
    "conv": rng.normal(0,0.1, size=(n_conv_layers, n_qubits, 3)),
    "final": rng.normal(0,0.1, size=(1, n_qubits, 3))
}

qnode = make_qcnn_qnode(n_qubits=n_qubits, n_conv_layers=n_conv_layers)

# -----------------------------
# Preprocess CNN Features for QCNN
# -----------------------------
def normalize_for_angle_embedding(X, mins=None, maxs=None):
    if mins is None:
        mins = X.min(axis=0, keepdims=True)
        maxs = X.max(axis=0, keepdims=True)
    denom = maxs - mins
    denom[denom == 0] = 1.0
    Xn = (X - mins)/denom
    return (Xn*2*np.pi) - np.pi, mins, maxs

def get_cnn_features(model, X):
    return model.predict(X, verbose=0)

# -----------------------------
# Batch QNode Evaluation
# -----------------------------
def qnode_batch_eval(qnode, X_inputs, weights, batch_size=8):
    q_feats = []
    for i in range(0, len(X_inputs), batch_size):
        batch = X_inputs[i : i + batch_size]
        batch_out = [qnode(x, weights) for x in batch]
        q_feats.append(np.vstack(batch_out))
    return np.vstack(q_feats)

# -----------------------------
# Prepare Data
# -----------------------------
# Assume X_train, X_test, y_train, y_test are preloaded images
# Convert to CNN features
X_train_cnn = get_cnn_features(cnn, X_train)
X_test_cnn = get_cnn_features(cnn, X_test)

# Normalize to [-pi, pi] for QCNN
X_train_qn, q_mins, q_maxs = normalize_for_angle_embedding(X_train_cnn)
X_test_qn, _, _ = normalize_for_angle_embedding(X_test_cnn, q_mins, q_maxs)

# -----------------------------
# Quantum Feature Extraction
# -----------------------------
print("⚙️ Computing quantum features...")
qtrain_feats = qnode_batch_eval(qnode, X_train_qn, weights, batch_size=8)
qtest_feats = qnode_batch_eval(qnode, X_test_qn, weights, batch_size=8)
print("✅ Done. Quantum feature shapes:", qtrain_feats.shape, qtest_feats.shape)

# -----------------------------
# Train SVM on Quantum Features
# -----------------------------
scaler = StandardScaler()
qtr_scaled = scaler.fit_transform(qtrain_feats)
qte_scaled = scaler.transform(qtest_feats)

svm_q = SVC(kernel="rbf", class_weight="balanced")
svm_q.fit(qtr_scaled, y_train)
yp_q = svm_q.predict(qte_scaled)

acc_q = accuracy_score(y_test, yp_q)
f1_q = f1_score(y_test, yp_q, average="macro")

print(f"\n🎯 CNN → QCNN → SVM Results:")
print(f"Accuracy = {acc_q:.3f}, F1-macro = {f1_q:.3f}")


In [ ]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.utils import to_categorical

# Assume X_train, X_test, y_train, y_test are loaded
# Reshape flat vectors into 32x32 grayscale images
X_train_img = X_train.reshape(-1, 32, 32, 1)
X_test_img  = X_test.reshape(-1, 32, 32, 1)

# One-hot encode labels if needed
num_classes = len(np.unique(y_train))
y_train_cat = to_categorical(y_train, num_classes)
y_test_cat  = to_categorical(y_test, num_classes)

# CNN model
cnn = Sequential([
    Conv2D(16, (3,3), activation='relu', input_shape=(32,32,1)),
    MaxPooling2D((2,2)),
    Conv2D(32, (3,3), activation='relu'),
    MaxPooling2D((2,2)),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])

cnn.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train
cnn.fit(X_train_img, y_train_cat, epochs=10, batch_size=32, validation_split=0.1)

# Evaluate
loss, acc = cnn.evaluate(X_test_img, y_test_cat)
print(f"✅ CNN Test Accuracy: {acc:.3f}")


In [ ]:
# =====================================================
# ⚡ Optimized 1D CNN + SVM for 1024D flattened data
# =====================================================

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

# -----------------------------
# Parameters
# -----------------------------
input_dim = 1024
n_classes = len(np.unique(y_train))
batch_size = 32
epochs = 5           # fewer epochs for fast training
dropout_rate = 0.2

# -----------------------------
# Reshape input for Conv1D
# -----------------------------
X_train_cnn = X_train.reshape(-1, input_dim, 1)
X_test_cnn  = X_test.reshape(-1, input_dim, 1)

# -----------------------------
# Build optimized 1D CNN
# -----------------------------
cnn = Sequential([
    Conv1D(16, kernel_size=5, activation='relu', input_shape=(input_dim,1)),  # smaller filter
    MaxPooling1D(pool_size=2),
    Conv1D(32, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(32, activation='relu'),  # smaller Dense layer
    Dropout(dropout_rate),
    Dense(n_classes, activation='softmax')
])

cnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# -----------------------------
# Train CNN quickly
# -----------------------------
cnn.fit(X_train_cnn, y_train, epochs=epochs, batch_size=batch_size, validation_split=0.1, verbose=1)

# -----------------------------
# Extract features from CNN
# -----------------------------
# Call the model once to build it and define input/output
_ = cnn(tf.zeros((1, input_dim, 1)))

feature_model = Model(inputs=cnn.input, outputs=cnn.layers[-3].output)  # Dense layer before Dropout
X_train_feats = feature_model.predict(X_train_cnn, verbose=0)
X_test_feats  = feature_model.predict(X_test_cnn, verbose=0)

# -----------------------------
# Scale features and train SVM
# -----------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_feats)
X_test_scaled  = scaler.transform(X_test_feats)

svm = SVC(kernel='rbf', class_weight='balanced')
svm.fit(X_train_scaled, y_train)
y_pred = svm.predict(X_test_scaled)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='macro')

print(f"\n🎯 Optimized CNN + SVM Results:")
print(f"Accuracy = {acc:.3f}, F1-macro = {f1:.3f}")

In [ ]:
# Build the model with input shape
cnn.build(input_shape=(None, input_dim, 1))  # None = batch size

from tensorflow.keras.models import Model
feature_model = Model(inputs=cnn.input, outputs=cnn.layers[-3].output)

# Now predict features
X_train_feats = feature_model.predict(X_train_cnn, verbose=0)
X_test_feats  = feature_model.predict(X_test_cnn, verbose=0)


In [ ]:
# =====================================================
# ⚡ CNN + Random Forest (Robust Version)
# =====================================================

import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

# -----------------------------
# Parameters
# -----------------------------
input_dim = 1024
n_classes = len(np.unique(y_train))
batch_size = 32
epochs = 10
dropout_rate = 0.2

# -----------------------------
# Reshape input for Conv1D
# -----------------------------
X_train_cnn = X_train.reshape(-1, input_dim, 1)
X_test_cnn  = X_test.reshape(-1, input_dim, 1)

# -----------------------------
# Build CNN with explicit Input layer
# -----------------------------
inputs = Input(shape=(input_dim, 1))
x = Conv1D(32, kernel_size=5, activation='relu')(inputs)
x = MaxPooling1D(pool_size=2)(x)
x = Conv1D(64, kernel_size=3, activation='relu')(x)
x = MaxPooling1D(pool_size=2)(x)
x = Flatten()(x)
feature_layer = Dense(64, activation='relu', name="feature_dense")(x)
x = Dropout(dropout_rate)(feature_layer)
outputs = Dense(n_classes, activation='softmax')(x)

cnn = Model(inputs=inputs, outputs=outputs)
cnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# -----------------------------
# Train CNN
# -----------------------------
cnn.fit(X_train_cnn, y_train, epochs=epochs, batch_size=batch_size, validation_split=0.1, verbose=1)

# -----------------------------
# Extract features from Dense layer
# -----------------------------
feature_model = Model(inputs=cnn.input, outputs=cnn.get_layer("feature_dense").output)
X_train_feats = feature_model.predict(X_train_cnn, verbose=0)
X_test_feats  = feature_model.predict(X_test_cnn, verbose=0)

# -----------------------------
# Scale features and train Random Forest
# -----------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_feats)
X_test_scaled  = scaler.transform(X_test_feats)

rf = RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42)
rf.fit(X_train_scaled, y_train)

# -----------------------------
# Evaluate
# -----------------------------
y_pred = rf.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='macro')

print(f"\n🎯 CNN + Random Forest Results:")
print(f"Accuracy = {acc:.3f}, F1-macro = {f1:.3f}")



In [ ]:
import numpy as np

# Example: 1024 → 32x32 grayscale
X_train_unflat = X_train.reshape(-1, 32, 32, 1)  # 1 channel for grayscale
X_test_unflat  = X_test.reshape(-1, 32, 32, 1)


In [ ]:
## =====================================================
# ✅ 1D CNN + SVM for 1024D flattened data (Fixed)
# =====================================================

import numpy as np
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

# -----------------------------
# Parameters
# -----------------------------
input_dim = 1024
n_classes = len(np.unique(y_train))

# -----------------------------
# Reshape input for Conv1D: (samples, time_steps, channels)
# -----------------------------
X_train_cnn = X_train.reshape(-1, input_dim, 1)
X_test_cnn  = X_test.reshape(-1, input_dim, 1)

# -----------------------------
# Build 1D CNN
# -----------------------------
cnn = Sequential([
    Conv1D(32, kernel_size=5, activation='relu', input_shape=(input_dim, 1)),
    MaxPooling1D(pool_size=2),
    Conv1D(64, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(64, activation='relu'),  # feature layer for SVM
    Dense(n_classes, activation='softmax')  # classification output
])

cnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# -----------------------------
# Train CNN
# -----------------------------
cnn.fit(X_train_cnn, y_train, epochs=10, batch_size=32, validation_split=0.1, verbose=1)

# -----------------------------
# Extract features from Dense layer before softmax
# -----------------------------
# Sequential model is now built, cnn.input is defined
feature_model = Model(inputs=cnn.input, outputs=cnn.layers[-2].output)  # second last layer
X_train_feats = feature_model.predict(X_train_cnn, verbose=0)
X_test_feats  = feature_model.predict(X_test_cnn, verbose=0)

# -----------------------------
# Scale features and train SVM
# -----------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_feats)
X_test_scaled  = scaler.transform(X_test_feats)

svm = SVC(kernel='rbf', class_weight='balanced')
svm.fit(X_train_scaled, y_train)
y_pred = svm.predict(X_test_scaled)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='macro')

print(f"\n🎯 CNN + SVM Results:")
print(f"Accuracy = {acc:.3f}, F1-macro = {f1:.3f}")



In [ ]:
X_train_img = X_train.reshape(-1, 32, 32, 1)
X_test_img  = X_test.reshape(-1, 32, 32, 1)


In [ ]:
# =====================================================
# ✅ 2D CNN for 32x32 grayscale images
# =====================================================

import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

# -----------------------------
# Parameters
# -----------------------------
img_height = 32
img_width = 32
channels = 1
n_classes = len(np.unique(y_train))
batch_size = 32
epochs = 20
dropout_rate = 0.3

# -----------------------------
# Unflatten your data
# -----------------------------
X_train_img = X_train.reshape(-1, img_height, img_width, channels)
X_test_img  = X_test.reshape(-1, img_height, img_width, channels)

# Normalize pixel values
X_train_img = X_train_img / 255.0
X_test_img  = X_test_img / 255.0

# One-hot encode labels
y_train_cat = to_categorical(y_train, n_classes)
y_test_cat  = to_categorical(y_test, n_classes)

# -----------------------------
# Build 2D CNN
# -----------------------------
cnn2d = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(img_height, img_width, channels)),
    MaxPooling2D((2,2)),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D((2,2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(dropout_rate),
    Dense(n_classes, activation='softmax')
])

cnn2d.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# -----------------------------
# Train 2D CNN
# -----------------------------
cnn2d.fit(X_train_img, y_train_cat, epochs=epochs, batch_size=batch_size,
          validation_split=0.1, verbose=1)

# -----------------------------
# Evaluate
# -----------------------------
y_pred_prob = cnn2d.predict(X_test_img)
y_pred = np.argmax(y_pred_prob, axis=1)

acc = accuracy_score(y_test)
from sklearn.metrics import accuracy_score, f1_score

# Evaluate
y_pred_prob = cnn2d.predict(X_test_img)
y_pred = np.argmax(y_pred_prob, axis=1)

acc = accuracy_score(y_test, y_pred)          # ✅ pass both true and predicted
f1 = f1_score(y_test, y_pred, average='macro')

print(f"\n🎯 2D CNN Results:")
print(f"Accuracy = {acc:.3f}, F1-macro = {f1:.3f}")


In [ ]:
# =====================================================
# ⚡ MLP + XGBoost on 1024D flattened data
# =====================================================

import numpy as np
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, Input
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score

# -----------------------------
# Parameters
# -----------------------------
input_dim = 1024
n_classes = len(np.unique(y_train))
batch_size = 32
epochs = 20
dropout_rate = 0.3

# -----------------------------
# Scale input
# -----------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# -----------------------------
# Build MLP feature extractor
# -----------------------------
mlp_input = Input(shape=(input_dim,))
x = Dense(512, activation='relu')(mlp_input)
x = Dropout(dropout_rate)(x)
x = Dense(256, activation='relu')(x)
feature_layer = Dense(128, activation='relu', name="mlp_features")(x)  # feature layer
mlp_model = Model(inputs=mlp_input, outputs=feature_layer)

mlp_model.compile(optimizer='adam', loss='mse')  # unsupervised features

# -----------------------------
# Extract features
# -----------------------------
X_train_feats = mlp_model.predict(X_train_scaled, verbose=0)
X_test_feats  = mlp_model.predict(X_test_scaled, verbose=0)

# -----------------------------
# Train XGBoost on MLP features
# -----------------------------
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    objective='multi:softmax',
    num_class=n_classes,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42
)

xgb.fit(X_train_feats, y_train)
y_pred = xgb.predict(X_test_feats)

# -----------------------------
# Evaluate
# -----------------------------
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='macro')

print(f"\n🎯 MLP + XGBoost Results:")
print(f"Accuracy = {acc:.3f}, F1-macro = {f1:.3f}")


In [ ]:
# =====================================================
# ⚛️ QCNN Encoder + SVM — Full DR Dataset (5 Classes)
# =====================================================

!pip install pennylane scikit-learn torch torchvision matplotlib seaborn tqdm

import os
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import pennylane as qml
from pennylane import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# -----------------------------
# 1️⃣ Dataset Setup
# -----------------------------
IMG_SIZE = 32  # QCNN-friendly size
BATCH = 16
DATA_DIR = "/content/diabeties"

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(),  # QCNN expects 1 channel
    transforms.ToTensor(),
])

train_ds = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), transform)
val_ds   = datasets.ImageFolder(os.path.join(DATA_DIR, "val"), transform)
test_ds  = datasets.ImageFolder(os.path.join(DATA_DIR, "test"), transform)

train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=False)
val_dl   = DataLoader(val_ds, batch_size=BATCH, shuffle=False)
test_dl  = DataLoader(test_ds, batch_size=BATCH, shuffle=False)

NUM_CLASSES = len(train_ds.classes)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------------
# 2️⃣ Flatten Images for PCA
# -----------------------------
def flatten_loader(dl):
    X, y = [], []
    for xb, yb in dl:
        X.append(xb.view(xb.size(0), -1))
        y.append(yb)
    X = torch.cat(X, dim=0).numpy()
    y = torch.cat(y, dim=0).numpy()
    return X, y

X_train, y_train = flatten_loader(train_dl)
X_val, y_val     = flatten_loader(val_dl)
X_test, y_test   = flatten_loader(test_dl)

print("Shapes:", X_train.shape, X_val.shape, X_test.shape)

# -----------------------------
# 3️⃣ PCA to Match Number of Qubits
# -----------------------------
n_qubits = 8
pca_q = PCA(n_components=n_qubits)
X_train_q = pca_q.fit_transform(X_train)
X_val_q   = pca_q.transform(X_val)
X_test_q  = pca_q.transform(X_test)

# Normalize to [-pi, pi]
def normalize_for_angle_embedding(X, mins=None, maxs=None):
    if mins is None:
        mins = X.min(axis=0, keepdims=True)
        maxs = X.max(axis=0, keepdims=True)
    denom = (maxs - mins)
    denom[denom == 0] = 1.0
    Xn = (X - mins) / denom
    return (Xn * 2 * np.pi) - np.pi, mins, maxs

X_train_qn, q_mins, q_maxs = normalize_for_angle_embedding(X_train_q)
X_val_qn, _, _ = normalize_for_angle_embedding(X_val_q, q_mins, q_maxs)
X_test_qn, _, _ = normalize_for_angle_embedding(X_test_q, q_mins, q_maxs)

# -----------------------------
# 4️⃣ QCNN Encoder
# -----------------------------
dev = qml.device("default.qubit", wires=n_qubits, shots=None)

def conv_block(weights, wires):
    for i, w in enumerate(wires):
        qml.RX(weights[i,0], wires=w)
        qml.RY(weights[i,1], wires=w)
        qml.RZ(weights[i,2], wires=w)
    for i in range(len(wires)-1):
        qml.CNOT(wires=[wires[i], wires[i+1]])

def pooling_block(wires):
    for i in range(0, len(wires), 2):
        a, b = wires[i], wires[(i+1)%len(wires)]
        qml.CRX(0.3, wires=[a,b])
        qml.CRY(0.1, wires=[a,b])
        qml.CNOT(wires=[b,a])

def make_qcnn_qnode(n_qubits, n_conv_layers=2):
    wires = list(range(n_qubits))
    @qml.qnode(dev, interface="autograd")
    def qnode(inputs, weights):
        qml.templates.AngleEmbedding(inputs, wires=wires)
        for l in range(n_conv_layers):
            conv_block(weights["conv"][l], wires)
            pooling_block(wires)
        qml.templates.StronglyEntanglingLayers(weights["final"], wires=wires)
        readout_wires = wires[:n_qubits//2]
        return [qml.expval(qml.PauliZ(i)) for i in readout_wires]
    return qnode

# Random weights
rng = np.random.default_rng(42)
weights = {
    "conv": rng.normal(0,0.1,size=(2,n_qubits,3)),  # 2 conv layers
    "final": rng.normal(0,0.1,size=(1,n_qubits,3))
}

qnode = make_qcnn_qnode(n_qubits=n_qubits)

# -----------------------------
# 5️⃣ Batch Feature Extraction
# -----------------------------
def qnode_batch_eval(qnode, X_inputs, weights, batch_size=16):
    feats = []
    for i in tqdm(range(0,len(X_inputs), batch_size), desc="QCNN feature extraction"):
        batch = X_inputs[i:i+batch_size]
        batch_out = [qnode(x, weights) for x in batch]
        feats.append(np.vstack(batch_out))
    return np.vstack(feats)

print("⚙️ Computing QCNN features...")
qtrain_feats = qnode_batch_eval(qnode, X_train_qn, weights)
qval_feats   = qnode_batch_eval(qnode, X_val_qn, weights)
qtest_feats  = qnode_batch_eval(qnode, X_test_qn, weights)
print("✅ Done!", qtrain_feats.shape, qval_feats.shape, qtest_feats.shape)

# -----------------------------
# 6️⃣ SVM Classifier
# -----------------------------
scaler = StandardScaler()
qtr_scaled = scaler.fit_transform(qtrain_feats)
qval_scaled = scaler.transform(qval_feats)
qte_scaled = scaler.transform(qtest_feats)

svm_q = SVC(kernel="rbf", class_weight="balanced")
svm_q.fit(qtr_scaled, y_train)

yp_val = svm_q.predict(qval_scaled)
yp_test = svm_q.predict(qte_scaled)

# -----------------------------
# 7️⃣ Metrics & Confusion Matrices
# -----------------------------
def print_metrics(y_true, y_pred, name="Dataset"):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    print(f"{name}: Accuracy={acc:.3f}, F1-macro={f1:.3f}")

def plot_cm(y_true, y_pred, title="Confusion Matrix"):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    cmn = cm / (cm.sum(axis=1, keepdims=True)+1e-9)
    plt.figure(figsize=(6,5))
    sns.heatmap(cmn, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=[str(i) for i in range(NUM_CLASSES)],
                yticklabels=[str(i) for i in range(NUM_CLASSES)])
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    plt.show()

print("\n🎯 QCNN + SVM Metrics — Validation Set")
print_metrics(y_val, yp_val, "Validation Set")
plot_cm(y_val, yp_val, "QCNN + SVM — Validation Set")

print("\n🎯 QCNN + SVM Metrics — Test Set")
print_metrics(y_test, yp_test, "Test Set")
plot_cm(y_test, yp_test, "QCNN + SVM — Test Set")


In [ ]:
# =====================================================
# ⚛️ QCNN Encoder + SVM — Full Dataset (5 Classes)
# =====================================================

import pennylane as qml
from pennylane import numpy as np
import numpy as onp
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# Quantum Device Setup
# -----------------------------
n_qubits = 8
dev = qml.device("default.qubit", wires=n_qubits, shots=None)

# -----------------------------
# QCNN Building Blocks
# -----------------------------
def conv_block(weights, wires):
    for i, w in enumerate(wires):
        qml.RX(weights[i, 0], wires=w)
        qml.RY(weights[i, 1], wires=w)
        qml.RZ(weights[i, 2], wires=w)
    for i in range(len(wires) - 1):
        qml.CNOT(wires=[wires[i], wires[i + 1]])

def pooling_block(wires):
    for i in range(0, len(wires), 2):
        a, b = wires[i], wires[(i + 1) % len(wires)]
        qml.CRX(0.3, wires=[a, b])
        qml.CRY(0.1, wires=[a, b])
        qml.CNOT(wires=[b, a])

# -----------------------------
# QCNN QNode
# -----------------------------
def make_qcnn_qnode(n_qubits, n_conv_layers=2):
    wires = list(range(n_qubits))

    @qml.qnode(dev, interface="autograd")
    def qnode(inputs, weights):
        qml.templates.AngleEmbedding(inputs, wires=wires)
        for l in range(n_conv_layers):
            conv_block(weights["conv"][l], wires)
            pooling_block(wires)
        qml.templates.StronglyEntanglingLayers(weights["final"], wires=wires)
        readout_wires = wires[: n_qubits // 2]
        return [qml.expval(qml.PauliZ(i)) for i in readout_wires]

    return qnode

# -----------------------------
# Initialize Random QCNN Weights
# -----------------------------
rng = np.random.default_rng(42)
weights = {
    "conv": rng.normal(0, 0.1, size=(2, n_qubits, 3)),  # 2 conv layers
    "final": rng.normal(0, 0.1, size=(1, n_qubits, 3)), # 1 final layer
}

qnode = make_qcnn_qnode(n_qubits=n_qubits)

# -----------------------------
# PCA for Quantum Inputs (8D)
# -----------------------------
pca_q = PCA(n_components=n_qubits)
X_train_q = pca_q.fit_transform(X_train)
X_val_q   = pca_q.transform(X_val)
X_test_q  = pca_q.transform(X_test)

# Normalize to [-π, π]
def normalize_for_angle_embedding(X, mins=None, maxs=None):
    if mins is None:
        mins = X.min(axis=0, keepdims=True)
        maxs = X.max(axis=0, keepdims=True)
    denom = (maxs - mins)
    denom[denom == 0] = 1.0
    Xn = (X - mins) / denom
    return (Xn * 2 * np.pi) - np.pi, mins, maxs

X_train_qn, q_mins, q_maxs = normalize_for_angle_embedding(X_train_q)
X_val_qn, _, _ = normalize_for_angle_embedding(X_val_q, q_mins, q_maxs)
X_test_qn, _, _ = normalize_for_angle_embedding(X_test_q, q_mins, q_maxs)

# -----------------------------
# Batch QCNN Feature Extraction
# -----------------------------
def qnode_batch_eval(qnode, X_inputs, weights, batch_size=16):
    q_feats = []
    for i in range(0, len(X_inputs), batch_size):
        batch = X_inputs[i : i + batch_size]
        batch_out = [qnode(x, weights) for x in batch]
        q_feats.append(np.vstack(batch_out))
    return np.vstack(q_feats)

print("⚙️ Computing quantum features for all images...")
qtrain_feats = qnode_batch_eval(qnode, X_train_qn, weights)
qval_feats   = qnode_batch_eval(qnode, X_val_qn, weights)
qtest_feats  = qnode_batch_eval(qnode, X_test_qn, weights)
print("✅ Done!")
print("Feature shapes:", qtrain_feats.shape, qval_feats.shape, qtest_feats.shape)

# -----------------------------
# Train SVM Classifier
# -----------------------------
scaler = StandardScaler()
qtr_scaled = scaler.fit_transform(qtrain_feats)
qval_scaled = scaler.transform(qval_feats)
qte_scaled = scaler.transform(qtest_feats)

svm_q = SVC(kernel="rbf", class_weight="balanced")
svm_q.fit(qtr_scaled, y_train)

yp_val = svm_q.predict(qval_scaled)
yp_test = svm_q.predict(qte_scaled)

# -----------------------------
# Metrics and Confusion Matrices
# -----------------------------
def print_metrics(y_true, y_pred, name="Dataset"):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    print(f"{name}: Accuracy={acc:.3f}, F1-macro={f1:.3f}")

def plot_cm(y_true, y_pred, title="Confusion Matrix"):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(5)))
    cmn = cm / (cm.sum(axis=1, keepdims=True) + 1e-9)
    plt.figure(figsize=(6,5))
    sns.heatmap(cmn, annot=True, fmt=".2f", cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    plt.show()

print("\n🎯 QCNN + SVM Metrics — Validation Set")
print_metrics(y_val, yp_val, "Validation Set")
plot_cm(y_val, yp_val, "QCNN + SVM — Validation Set")

print("\n🎯 QCNN + SVM Metrics — Test Set")
print_metrics(y_test, yp_test, "Test Set")
plot_cm(y_test, yp_test, "QCNN + SVM — Test Set")


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

IMG_SIZE = 32  # or whatever size you are using
BATCH = 16

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(),  # QCNN expects 1 channel
    transforms.ToTensor(),
])

val_ds = datasets.ImageFolder("/content/diabeties/val", transform)
val_dl = DataLoader(val_ds, batch_size=BATCH, shuffle=False)


In [ ]:
import torch
import numpy as np

def flatten_loader(dl):
    X, y = [], []
    for xb, yb in dl:
        X.append(xb.view(xb.size(0), -1))
        y.append(yb)
    X = torch.cat(X, dim=0).numpy()
    y = torch.cat(y, dim=0).numpy()
    return X, y

X_val, y_val = flatten_loader(val_dl)


In [ ]:
pca_q = PCA(n_components=n_qubits)
X_train_q = pca_q.fit_transform(X_train)
X_val_q   = pca_q.transform(X_val)
X_test_q  = pca_q.transform(X_test)

# Normalize to [-pi, pi] for AngleEmbedding
def normalize_for_angle_embedding(X, mins=None, maxs=None):
    if mins is None:
        mins = X.min(axis=0, keepdims=True)
        maxs = X.max(axis=0, keepdims=True)
    denom = (maxs - mins)
    denom[denom == 0] = 1.0
    Xn = (X - mins) / denom
    return (Xn * 2 * np.pi) - np.pi, mins, maxs

X_train_qn, q_mins, q_maxs = normalize_for_angle_embedding(X_train_q)
X_val_qn, _, _ = normalize_for_angle_embedding(X_val_q, q_mins, q_maxs)
X_test_qn, _, _ = normalize_for_angle_embedding(X_test_q, q_mins, q_maxs)


In [ ]:
# =====================================================
# ⚛️ QCNN Encoder + Classical SVM — Full Evaluation
# =====================================================

import pennylane as qml
from pennylane import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as onp

# -----------------------------
# Device & parameters
# -----------------------------
n_qubits = 8
dev = qml.device("default.qubit", wires=n_qubits, shots=None)

def conv_block(weights, wires):
    for i, w in enumerate(wires):
        qml.RX(weights[i, 0], wires=w)
        qml.RY(weights[i, 1], wires=w)
        qml.RZ(weights[i, 2], wires=w)
    for i in range(len(wires) - 1):
        qml.CNOT(wires=[wires[i], wires[i + 1]])

def pooling_block(wires):
    for i in range(0, len(wires), 2):
        a, b = wires[i], wires[(i + 1) % len(wires)]
        qml.CRX(0.3, wires=[a, b])
        qml.CRY(0.1, wires=[a, b])
        qml.CNOT(wires=[b, a])

def make_qcnn_qnode(n_qubits, n_conv_layers=2):
    wires = list(range(n_qubits))

    @qml.qnode(dev, interface="autograd")
    def qnode(inputs, weights):
        qml.templates.AngleEmbedding(inputs, wires=wires)
        for l in range(n_conv_layers):
            conv_block(weights["conv"][l], wires)
            pooling_block(wires)
        qml.templates.StronglyEntanglingLayers(weights["final"], wires=wires)
        readout_wires = wires[: n_qubits // 2]
        return [qml.expval(qml.PauliZ(i)) for i in readout_wires]

    return qnode

# -----------------------------
# Initialize QCNN weights
# -----------------------------
rng = np.random.default_rng(42)
weights = {
    "conv": rng.normal(0, 0.1, size=(2, n_qubits, 3)),
    "final": rng.normal(0, 0.1, size=(1, n_qubits, 3)),
}

qnode = make_qcnn_qnode(n_qubits=n_qubits, n_conv_layers=2)

# -----------------------------
# Reduce features to 8D (for quantum encoding)
# -----------------------------
pca_q = PCA(n_components=8, random_state=42)
X_train_q = pca_q.fit_transform(X_train)
X_test_q  = pca_q.transform(X_test)
X_eval_q  = pca_q.transform(X_eval)

def normalize_for_angle_embedding(X, mins=None, maxs=None):
    if mins is None:
        mins = X.min(axis=0, keepdims=True)
        maxs = X.max(axis=0, keepdims=True)
    denom = (maxs - mins)
    denom[denom == 0] = 1.0
    Xn = (X - mins) / denom
    return (Xn * 2 * np.pi) - np.pi, mins, maxs

X_train_qn, q_mins, q_maxs = normalize_for_angle_embedding(X_train_q)
X_test_qn, _, _  = normalize_for_angle_embedding(X_test_q, q_mins, q_maxs)
X_eval_qn, _, _  = normalize_for_angle_embedding(X_eval_q, q_mins, q_maxs)

# -----------------------------
# Quantum feature extraction
# -----------------------------
def qnode_batch_eval(qnode, X_inputs, weights, batch_size=16):
    q_feats = []
    for i in range(0, len(X_inputs), batch_size):
        batch = X_inputs[i : i + batch_size]
        batch_out = [qnode(x, weights) for x in batch]
        q_feats.append(np.vstack(batch_out))
    return np.vstack(q_feats)

print("⚙️ Computing quantum features for all DR images...")
qtrain_feats = qnode_batch_eval(qnode, X_train_qn, weights, batch_size=16)
qtest_feats  = qnode_batch_eval(qnode, X_test_qn,  weights, batch_size=16)
qeval_feats  = qnode_batch_eval(qnode, X_eval_qn,  weights, batch_size=16)
print("✅ Done! Feature shapes:", qtrain_feats.shape, qtest_feats.shape, qeval_feats.shape)

# -----------------------------
# Train & evaluate SVM
# -----------------------------
scaler = StandardScaler()
qtr_scaled = scaler.fit_transform(qtrain_feats)
qte_scaled = scaler.transform(qtest_feats)
qev_scaled = scaler.transform(qeval_feats)

svm_q = SVC(kernel="rbf", class_weight="balanced")
svm_q.fit(qtr_scaled, y_train)
yp_test = svm_q.predict(qte_scaled)
yp_eval = svm_q.predict(qev_scaled)

# -----------------------------
# Metrics & Confusion Matrices
# -----------------------------
def plot_cm(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    cmn = cm / (cm.sum(axis=1, keepdims=True) + 1e-9)
    plt.figure(figsize=(6,5))
    sns.heatmap(cmn, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=[str(i) for i in range(NUM_CLASSES)],
                yticklabels=[str(i) for i in range(NUM_CLASSES)])
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

def print_metrics(y_true, y_pred, name):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')
    kappa = cohen_kappa_score(y_true, y_pred)
    print(f"{name}: Accuracy={acc:.3f}, F1-macro={f1:.3f}, Cohen-Kappa={kappa:.3f}")

print("\n🎯 Results on Full DR Dataset (All Images):")
print_metrics(y_test, yp_test, "QCNN + SVM (Test)")
plot_cm(y_test, yp_test, "QCNN + SVM — Test Set")

print_metrics(y_eval, yp_eval, "QCNN + SVM (Eval)")
plot_cm(y_eval, yp_eval, "QCNN + SVM — Eval Set")


In [ ]:
# %% [markdown]
# Confusion Matrices & Metrics for Best Quantum vs Best Classical

# %% [code]
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, cohen_kappa_score

# Ensure these are defined
# class_names = ['0', '1', '2', '3', '4']
# NUM_CLASSES = len(class_names)

def plot_cm(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    cmn = cm / (cm.sum(axis=1, keepdims=True) + 1e-9)
    plt.figure(figsize=(5,4))
    sns.heatmap(cmn, annot=True, fmt=".2f", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

def print_metrics(y_true, y_pred, name):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')
    kappa = cohen_kappa_score(y_true, y_pred)
    print(f"{name}: Accuracy={acc:.3f}, F1-macro={f1:.3f}, Cohen-Kappa={kappa:.3f}")

# --- Pick best classical model (higher F1-macro) ---
if f1_score(y_test, mlp_pred, average='macro') >= f1_score(y_test, svm_pred, average='macro'):
    best_classical_pred = mlp_pred
    best_classical_name = "CNN + MLP"
else:
    best_classical_pred = svm_pred
    best_classical_name = "CNN + SVM"

# --- Metrics ---
print_metrics(y_test, best_classical_pred, best_classical_name)

# If QSVM used a subset of test samples (like in your pipeline)
y_test_qs = y_test[:len(qs_pred)]
print_metrics(y_test_qs, qs_pred, "QCNN + QSVM")

# --- Confusion Matrices ---
plot_cm(y_test, best_classical_pred, f"{best_classical_name} — Confusion Matrix")
plot_cm(y_test_qs, qs_pred, "QCNN + QSVM — Confusion Matrix")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
import cv2
import os
from glob import glob
from tqdm import tqdm
import zipfile
import pickle
import warnings
warnings.filterwarnings('ignore')

# TensorFlow/Keras imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import (
    VGG16, ResNet50, InceptionV3, EfficientNetB0, MobileNetV2
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Mount Google Drive (if needed)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except:
    print("Not running in Colab, skipping drive mount")

# Configuration
CONFIG = {
    'IMG_SIZE': (224, 224),  # Standard size for pre-trained models
    'DATA_DIR': '/content/diabeties',
    'ZIP_PATH': '/content/drive/MyDrive/diabeties.zip',
    'EXTRACT_PATH': '/content/',
    'BATCH_SIZE': 32,
    'RANDOM_STATE': 42,
    'CLASS_NAMES': {
        0: 'No DR',
        1: 'Mild',
        2: 'Moderate',
        3: 'Severe',
        4: 'Proliferative DR'
    },
    # CNN Feature Extractor Options
    'FEATURE_EXTRACTOR': 'EfficientNetB0',  # Options: VGG16, ResNet50, InceptionV3, EfficientNetB0, MobileNetV2
    'USE_AUGMENTATION': True,
    'FINE_TUNE': False  # Whether to fine-tune the CNN or just use as feature extractor
}


class CNNFeatureExtractor:
    """Extract features from images using pre-trained CNN models"""

    def __init__(self, model_name='EfficientNetB0', input_shape=(224, 224, 3)):
        self.model_name = model_name
        self.input_shape = input_shape
        self.feature_extractor = None
        self.build_feature_extractor()

    def build_feature_extractor(self):
        """Build feature extractor from pre-trained models"""
        print(f"\nBuilding feature extractor using {self.model_name}...")

        # Load pre-trained model without top classification layer
        if self.model_name == 'VGG16':
            base_model = VGG16(weights='imagenet', include_top=False, input_shape=self.input_shape)
        elif self.model_name == 'ResNet50':
            base_model = ResNet50(weights='imagenet', include_top=False, input_shape=self.input_shape)
        elif self.model_name == 'InceptionV3':
            base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=self.input_shape)
        elif self.model_name == 'EfficientNetB0':
            base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=self.input_shape)
        elif self.model_name == 'MobileNetV2':
            base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=self.input_shape)
        else:
            raise ValueError(f"Unknown model: {self.model_name}")

        # Freeze base model layers
        base_model.trainable = False

        # Add global average pooling to get fixed-size feature vectors
        x = base_model.output
        x = GlobalAveragePooling2D()(x)

        # Create the feature extractor model
        self.feature_extractor = Model(inputs=base_model.input, outputs=x)

        print(f"Feature extractor built. Output shape: {self.feature_extractor.output_shape}")
        print(f"Number of features: {self.feature_extractor.output_shape[1]}")

    def preprocess_image(self, img):
        """Preprocess image for the specific CNN model"""
        # Convert to RGB if needed
        if len(img.shape) == 2:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

        # Resize to target size
        img = cv2.resize(img, self.input_shape[:2])

        # Model-specific preprocessing
        if self.model_name == 'VGG16':
            from tensorflow.keras.applications.vgg16 import preprocess_input
        elif self.model_name == 'ResNet50':
            from tensorflow.keras.applications.resnet50 import preprocess_input
        elif self.model_name == 'InceptionV3':
            from tensorflow.keras.applications.inception_v3 import preprocess_input
        elif self.model_name == 'EfficientNetB0':
            from tensorflow.keras.applications.efficientnet import preprocess_input
        elif self.model_name == 'MobileNetV2':
            from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

        img = preprocess_input(img)
        return img

    def extract_features(self, images, batch_size=32):
        """Extract features from a batch of images"""
        print(f"\nExtracting features from {len(images)} images...")
        features = []

        for i in tqdm(range(0, len(images), batch_size)):
            batch = images[i:i+batch_size]
            batch_features = self.feature_extractor.predict(batch, verbose=0)
            features.append(batch_features)

        features = np.vstack(features)
        print(f"Extracted features shape: {features.shape}")
        return features


class DRClassifierWithCNN:
    """Diabetic Retinopathy Classifier using CNN features"""

    def __init__(self, config):
        self.config = config
        self.feature_extractor = None
        self.scaler = StandardScaler()
        self.rf_model = None
        self.svm_model = None
        self.lr_model = None

    def extract_dataset(self):
        """Extract zip file if needed"""
        if os.path.exists(self.config['DATA_DIR']):
            print(f"Dataset already extracted at {self.config['DATA_DIR']}")
            return

        print("Extracting dataset...")
        with zipfile.ZipFile(self.config['ZIP_PATH'], 'r') as zip_ref:
            zip_ref.extractall(self.config['EXTRACT_PATH'])
        print("Extraction complete!")

    def load_images_from_split(self, split):
        """Load images and labels from a specific split"""
        images = []
        labels = []

        split_path = os.path.join(self.config['DATA_DIR'], split)
        if not os.path.exists(split_path):
            print(f"Warning: {split_path} does not exist!")
            return np.array(images), np.array(labels)

        subdirs = sorted([d for d in os.listdir(split_path)
                         if os.path.isdir(os.path.join(split_path, d))])

        for subdir in subdirs:
            class_folder = os.path.join(split_path, subdir)

            # Extract class number
            class_num = None
            for i in range(5):
                if f'class{i}' in subdir.lower():
                    class_num = i
                    break

            if class_num is None:
                continue

            # Get image files
            extensions = ['*.jpg', '*.png', '*.jpeg', '*.JPG', '*.PNG', '*.JPEG']
            image_files = []
            for ext in extensions:
                image_files.extend(glob(os.path.join(class_folder, ext)))

            print(f"Loading {len(image_files)} images from {split}/{subdir} (Class {class_num})")

            for img_path in tqdm(image_files, desc=f"{split}-class{class_num}"):
                try:
                    img = cv2.imread(img_path)
                    if img is None:
                        continue

                    # Convert BGR to RGB
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

                    # Apply preprocessing for better retinal image quality
                    img = self._preprocess_retinal_image(img)

                    images.append(img)
                    labels.append(class_num)

                except Exception as e:
                    print(f"Error loading {img_path}: {e}")

        return np.array(images), np.array(labels)

    def _preprocess_retinal_image(self, img):
        """Enhanced preprocessing for retinal images"""
        # Apply CLAHE for better contrast
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        l = clahe.apply(l)
        img = cv2.merge([l, a, b])
        img = cv2.cvtColor(img, cv2.COLOR_LAB2RGB)

        # Gaussian blur to reduce noise
        img = cv2.GaussianBlur(img, (3, 3), 0)

        return img

    def load_data(self):
        """Load all splits"""
        print("="*60)
        print("LOADING DATASET")
        print("="*60)

        print("\nLoading TRAIN images...")
        self.X_train, self.y_train = self.load_images_from_split('train')

        print("\nLoading TEST images...")
        self.X_test, self.y_test = self.load_images_from_split('test')

        print("\nLoading VALIDATION images...")
        self.X_val, self.y_val = self.load_images_from_split('val')

        # Validate data
        if len(self.X_train) == 0 or len(self.X_test) == 0 or len(self.X_val) == 0:
            raise ValueError("One or more datasets are empty!")

        print(f"\nDataset loaded successfully:")
        print(f"Train: {self.X_train.shape}, Labels: {self.y_train.shape}")
        print(f"Test: {self.X_test.shape}, Labels: {self.y_test.shape}")
        print(f"Val: {self.X_val.shape}, Labels: {self.y_val.shape}")

        self._print_class_distribution()

    def _print_class_distribution(self):
        """Print class distribution"""
        print("\n" + "="*60)
        print("CLASS DISTRIBUTION")
        print("="*60)
        for name, labels in [('Train', self.y_train), ('Test', self.y_test), ('Val', self.y_val)]:
            counts = np.bincount(labels, minlength=5)
            total = len(labels)
            print(f"\n{name} ({total} images):")
            for cls, count in enumerate(counts):
                cls_name = self.config['CLASS_NAMES'][cls]
                percentage = (count / total * 100) if total > 0 else 0
                print(f"  {cls_name:20s}: {count:5d} ({percentage:5.1f}%)")

    def extract_cnn_features(self):
        """Extract features using pre-trained CNN"""
        print("\n" + "="*60)
        print("EXTRACTING CNN FEATURES")
        print("="*60)

        # Initialize feature extractor
        self.feature_extractor = CNNFeatureExtractor(
            model_name=self.config['FEATURE_EXTRACTOR'],
            input_shape=(*self.config['IMG_SIZE'], 3)
        )

        # Preprocess images for CNN
        print("\nPreprocessing images for CNN...")
        X_train_preprocessed = np.array([
            self.feature_extractor.preprocess_image(img) for img in tqdm(self.X_train, desc="Train")
        ])
        X_test_preprocessed = np.array([
            self.feature_extractor.preprocess_image(img) for img in tqdm(self.X_test, desc="Test")
        ])
        X_val_preprocessed = np.array([
            self.feature_extractor.preprocess_image(img) for img in tqdm(self.X_val, desc="Val")
        ])

        # Extract features
        self.X_train_features = self.feature_extractor.extract_features(
            X_train_preprocessed, batch_size=self.config['BATCH_SIZE']
        )
        self.X_test_features = self.feature_extractor.extract_features(
            X_test_preprocessed, batch_size=self.config['BATCH_SIZE']
        )
        self.X_val_features = self.feature_extractor.extract_features(
            X_val_preprocessed, batch_size=self.config['BATCH_SIZE']
        )

        # Scale features
        print("\nScaling features...")
        self.X_train_scaled = self.scaler.fit_transform(self.X_train_features)
        self.X_test_scaled = self.scaler.transform(self.X_test_features)
        self.X_val_scaled = self.scaler.transform(self.X_val_features)

        print(f"\nFeature extraction complete!")
        print(f"Feature dimensions: {self.X_train_scaled.shape[1]}")

    def train_random_forest(self, n_estimators=200):
        """Train Random Forest on CNN features"""
        print("\n" + "="*60)
        print("TRAINING RANDOM FOREST")
        print("="*60)

        self.rf_model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=30,
            min_samples_split=2,
            min_samples_leaf=1,
            random_state=self.config['RANDOM_STATE'],
            n_jobs=-1,
            verbose=1
        )

        self.rf_model.fit(self.X_train_scaled, self.y_train)
        self._evaluate_model(self.rf_model, "Random Forest")

    def train_svm(self, C=10.0):
        """Train SVM on CNN features"""
        print("\n" + "="*60)
        print("TRAINING SVM")
        print("="*60)

        self.svm_model = SVC(
            kernel='rbf',
            C=C,
            gamma='scale',
            probability=True,
            random_state=self.config['RANDOM_STATE'],
            verbose=True
        )

        self.svm_model.fit(self.X_train_scaled, self.y_train)
        self._evaluate_model(self.svm_model, "SVM")

    def train_logistic_regression(self, C=1.0):
        """Train Logistic Regression on CNN features"""
        print("\n" + "="*60)
        print("TRAINING LOGISTIC REGRESSION")
        print("="*60)

        self.lr_model = LogisticRegression(
            C=C,
            max_iter=1000,
            multi_class='multinomial',
            random_state=self.config['RANDOM_STATE'],
            n_jobs=-1,
            verbose=1
        )

        self.lr_model.fit(self.X_train_scaled, self.y_train)
        self._evaluate_model(self.lr_model, "Logistic Regression")

    def _evaluate_model(self, model, model_name):
        """Comprehensive model evaluation"""
        # Predictions
        y_pred_test = model.predict(self.X_test_scaled)
        y_pred_val = model.predict(self.X_val_scaled)

        # Store predictions for later visualization
        if not hasattr(self, 'predictions'):
            self.predictions = {}
        self.predictions[model_name] = {
            'test': y_pred_test,
            'val': y_pred_val
        }

        # Metrics
        print(f"\n{'='*60}")
        print(f"{model_name.upper()} RESULTS")
        print(f"{'='*60}")

        print(f"\nTest Set:")
        print(f"  Accuracy:  {accuracy_score(self.y_test, y_pred_test):.4f}")
        print(f"  F1-Score:  {f1_score(self.y_test, y_pred_test, average='weighted'):.4f}")

        print(f"\nValidation Set:")
        print(f"  Accuracy:  {accuracy_score(self.y_val, y_pred_val):.4f}")
        print(f"  F1-Score:  {f1_score(self.y_val, y_pred_val, average='weighted'):.4f}")

        # Classification reports
        unique_classes = np.unique(self.y_train)
        class_names = [self.config['CLASS_NAMES'][i] for i in unique_classes]

        print(f"\nTest Classification Report:")
        print(classification_report(self.y_test, y_pred_test,
                                   labels=unique_classes,
                                   target_names=class_names,
                                   zero_division=0))

    def visualize_results(self):
        """Create comprehensive visualizations"""
        if not hasattr(self, 'predictions'):
            print("No predictions available. Train models first.")
            return

        n_models = len(self.predictions)
        fig, axes = plt.subplots(n_models, 2, figsize=(14, 6*n_models))

        if n_models == 1:
            axes = axes.reshape(1, -1)

        for idx, (model_name, preds) in enumerate(self.predictions.items()):
            # Test set confusion matrix
            cm_test = confusion_matrix(self.y_test, preds['test'])
            sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues', ax=axes[idx, 0])
            axes[idx, 0].set_title(f'{model_name} - Test Set')
            axes[idx, 0].set_xlabel('Predicted')
            axes[idx, 0].set_ylabel('Actual')

            # Validation set confusion matrix
            cm_val = confusion_matrix(self.y_val, preds['val'])
            sns.heatmap(cm_val, annot=True, fmt='d', cmap='Greens', ax=axes[idx, 1])
            axes[idx, 1].set_title(f'{model_name} - Validation Set')
            axes[idx, 1].set_xlabel('Predicted')
            axes[idx, 1].set_ylabel('Actual')

        plt.tight_layout()
        plt.savefig('cnn_features_confusion_matrices.png', dpi=300, bbox_inches='tight')
        plt.show()

        # Model comparison
        self._plot_model_comparison()

    def _plot_model_comparison(self):
        """Plot model comparison"""
        if not hasattr(self, 'predictions'):
            return

        results_data = []
        for model_name, preds in self.predictions.items():
            results_data.extend([
                {
                    'Model': model_name,
                    'Dataset': 'Test',
                    'Accuracy': accuracy_score(self.y_test, preds['test']),
                    'F1-Score': f1_score(self.y_test, preds['test'], average='weighted')
                },
                {
                    'Model': model_name,
                    'Dataset': 'Validation',
                    'Accuracy': accuracy_score(self.y_val, preds['val']),
                    'F1-Score': f1_score(self.y_val, preds['val'], average='weighted')
                }
            ])

        results = pd.DataFrame(results_data)

        print("\n" + "="*60)
        print("MODEL COMPARISON (CNN FEATURES)")
        print("="*60)
        print(results.to_string(index=False))

        # Bar plots
        fig, ax = plt.subplots(1, 2, figsize=(14, 5))

        results.pivot(index='Dataset', columns='Model', values='Accuracy').plot(kind='bar', ax=ax[0])
        ax[0].set_title(f'Model Accuracy Comparison\n(Features: {self.config["FEATURE_EXTRACTOR"]})')
        ax[0].set_ylabel('Accuracy')
        ax[0].legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')
        ax[0].set_ylim([0, 1])
        ax[0].set_xticklabels(ax[0].get_xticklabels(), rotation=0)

        results.pivot(index='Dataset', columns='Model', values='F1-Score').plot(kind='bar', ax=ax[1])
        ax[1].set_title(f'Model F1-Score Comparison\n(Features: {self.config["FEATURE_EXTRACTOR"]})')
        ax[1].set_ylabel('F1-Score')
        ax[1].legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')
        ax[1].set_ylim([0, 1])
        ax[1].set_xticklabels(ax[1].get_xticklabels(), rotation=0)

        plt.tight_layout()
        plt.savefig('cnn_features_model_comparison.png', dpi=300, bbox_inches='tight')
        plt.show()

    def save_models(self, output_dir='models_cnn_features'):
        """Save all models and preprocessing objects"""
        os.makedirs(output_dir, exist_ok=True)

        if self.rf_model:
            with open(f'{output_dir}/random_forest_cnn.pkl', 'wb') as f:
                pickle.dump(self.rf_model, f)

        if self.svm_model:
            with open(f'{output_dir}/svm_cnn.pkl', 'wb') as f:
                pickle.dump(self.svm_model, f)

        if self.lr_model:
            with open(f'{output_dir}/logistic_regression_cnn.pkl', 'wb') as f:
                pickle.dump(self.lr_model, f)

        with open(f'{output_dir}/scaler_cnn.pkl', 'wb') as f:
            pickle.dump(self.scaler, f)

        # Save feature extractor configuration
        with open(f'{output_dir}/feature_extractor_config.pkl', 'wb') as f:
            pickle.dump({
                'model_name': self.config['FEATURE_EXTRACTOR'],
                'input_shape': (*self.config['IMG_SIZE'], 3)
            }, f)

        print(f"\nModels saved to {output_dir}/ directory!")

    def predict_image(self, image_path, model_type='rf'):
        """Predict the class of a new retina image"""
        # Select model
        if model_type == 'rf':
            model = self.rf_model
        elif model_type == 'svm':
            model = self.svm_model
        elif model_type == 'lr':
            model = self.lr_model
        else:
            raise ValueError(f"Unknown model type: {model_type}")

        if model is None:
            raise ValueError(f"Model {model_type} not trained yet!")

        # Load and preprocess image
        img = cv2.imread(image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = self._preprocess_retinal_image(img)

        # Extract CNN features
        img_preprocessed = self.feature_extractor.preprocess_image(img)
        img_preprocessed = np.expand_dims(img_preprocessed, axis=0)
        features = self.feature_extractor.feature_extractor.predict(img_preprocessed, verbose=0)
        features_scaled = self.scaler.transform(features)

        # Predict
        prediction = model.predict(features_scaled)[0]
        severity = self.config['CLASS_NAMES'].get(prediction, f'Class {prediction}')

        # Get probability if available
        if hasattr(model, 'predict_proba'):
            proba = model.predict_proba(features_scaled)[0]
            confidence = proba[prediction]
            return prediction, severity, confidence, proba

        return prediction, severity, None, None


# Main execution
if __name__ == "__main__":
    print("="*60)
    print("DIABETIC RETINOPATHY CLASSIFICATION")
    print("Using CNN as Feature Extractor")
    print("="*60)
    print(f"\nFeature Extractor: {CONFIG['FEATURE_EXTRACTOR']}")
    print(f"Image Size: {CONFIG['IMG_SIZE']}")

    # Initialize classifier
    classifier = DRClassifierWithCNN(CONFIG)

    # Extract dataset
    classifier.extract_dataset()

    # Load data
    classifier.load_data()

    # Extract CNN features
    classifier.extract_cnn_features()

    # Train models
    print("\n" + "="*60)
    print("TRAINING PHASE")
    print("="*60)

    classifier.train_random_forest(n_estimators=200)
    classifier.train_svm(C=10.0)
    classifier.train_logistic_regression(C=1.0)

    # Visualize results
    classifier.visualize_results()

    # Save models
    classifier.save_models()

    print("\n" + "="*60)
    print("TRAINING COMPLETE!")
    print("="*60)

    # Example prediction
    # pred_class, severity, confidence, proba = classifier.predict_image(
    #     'test_image.jpg', model_type='rf'
    # )
    # print(f"\nPrediction: {severity} (Class {pred_class})")
    # print(f"Confidence: {confidence:.2%}")
    # print(f"All probabilities: {proba}")

In [ ]:
{
  "nbformat": 4,
  "nbformat_minor": 0,
  "metadata": {
    "colab": {
      "provenance": []
    },
    "kernelspec": {
      "name": "python3",
      "display_name": "Python 3"
    },
    "language_info": {
      "name": "python"
    }
  },
  "cells": [
    {
      "cell_type": "markdown",
      "source": [
        "**Installations and imports **"
      ],
      "metadata": {
        "id": "UCp_XscDWjS-"
      }
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {
        "id": "9XjctF4XWcTb"
      },
      "outputs": [],
      "source": [
        "# %% [markdown]\n",
        "# Setup\n",
        "\n",
        "# %% [code]\n",
        "!pip -q install timm==1.0.9 pennylane==0.39.0 pennylane-lightning==0.39.0 scikit-learn==1.5.1 albumentations==1.4.18 kornia==0.7.4\n",
        "\n",
        "import os, json, math, time, random, shutil, gc, warnings\n",
        "warnings.filterwarnings(\"ignore\")\n",
        "\n",
        "import numpy as np\n",
        "import pandas as pd\n",
        "\n",
        "import torch\n",
        "import torch.nn as nn\n",
        "import torch.nn.functional as F\n",
        "from torch.utils.data import DataLoader\n",
        "from torch.optim import AdamW\n",
        "from torch.optim.lr_scheduler import ReduceLROnPlateau\n",
        "\n",
        "import torchvision as tv\n",
        "from torchvision import transforms\n",
        "from torchvision.datasets import ImageFolder\n",
        "\n",
        "import timm\n",
        "\n",
        "from sklearn.svm import SVC\n",
        "from sklearn.decomposition import PCA\n",
        "from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, cohen_kappa_score\n",
        "from sklearn.preprocessing import StandardScaler\n",
        "from sklearn.utils.class_weight import compute_class_weight\n",
        "\n",
        "import pennylane as qml\n",
        "from pennylane import numpy as pnp\n",
        "\n",
        "DEVICE = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n",
        "SEED   = 42\n",
        "random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)\n",
        "DEVICE\n"
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "# %% [markdown]\n",
        "# Mount Google Drive (Colab) and set dataset paths\n",
        "\n",
        "# %% [code]\n",
        "from google.colab import drive\n",
        "drive.mount('/content/drive', force_remount=True)\n",
        "\n",
        "ROOT = \"/content/drive/MyDrive/main\"  # <-- change if needed\n",
        "TRAIN_DIR = os.path.join(ROOT, \"train\")\n",
        "TEST_DIR  = os.path.join(ROOT, \"test\")\n",
        "\n",
        "assert os.path.isdir(TRAIN_DIR) and os.path.isdir(TEST_DIR), \"Check your 'main/train' and 'main/test' paths.\"\n",
        "print(\"Train:\", TRAIN_DIR)\n",
        "print(\"Test :\", TEST_DIR)\n"
      ],
      "metadata": {
        "id": "Tju8aC30WqPN"
      },
      "execution_count": null,
      "outputs": []
    },
    {
      "cell_type": "code",
      "source": [
        "# %% [markdown]\n",
        "# Data transforms, loaders, and class weights (handles imbalance)\n",
        "\n",
        "# %% [code]\n",
        "IMG_SIZE = 300  # EfficientNet-B3 default is 300\n",
        "BATCH_TRAIN = 32\n",
        "BATCH_EVAL  = 64\n",
        "NUM_CLASSES = 5\n",
        "\n",
        "# Medical-safe augments\n",
        "train_tfms = transforms.Compose([\n",
        "    transforms.Resize((IMG_SIZE, IMG_SIZE)),\n",
        "    transforms.RandomHorizontalFlip(p=0.5),\n",
        "    transforms.RandomVerticalFlip(p=0.15),\n",
        "    transforms.RandomRotation(15),\n",
        "    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.05, hue=0.02),\n",
        "    transforms.ToTensor(),\n",
        "    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),\n",
        "])\n",
        "\n",
        "test_tfms = transforms.Compose([\n",
        "    transforms.Resize((IMG_SIZE, IMG_SIZE)),\n",
        "    transforms.ToTensor(),\n",
        "    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),\n",
        "])\n",
        "\n",
        "ds_train = ImageFolder(TRAIN_DIR, transform=train_tfms)\n",
        "ds_test  = ImageFolder(TEST_DIR,  transform=test_tfms)\n",
        "class_names = ds_train.classes\n",
        "print(\"Classes:\", class_names)\n",
        "\n",
        "# Class weights for imbalance\n",
        "y_train_labels = np.array([ds_train.samples[i][1] for i in range(len(ds_train))])\n",
        "cls_weights = compute_class_weight(class_weight=\"balanced\", classes=np.arange(NUM_CLASSES), y=y_train_labels)\n",
        "cls_weights = torch.tensor(cls_weights, dtype=torch.float32).to(DEVICE)\n",
        "print(\"Class weights:\", cls_weights.tolist())\n",
        "\n",
        "dl_train = DataLoader(ds_train, batch_size=BATCH_TRAIN, shuffle=True, num_workers=2, pin_memory=True)\n",
        "dl_test  = DataLoader(ds_test,  batch_size=BATCH_EVAL,  shuffle=False, num_workers=2, pin_memory=True)\n"
      ],
      "metadata": {
        "id": "RY-E680VWrP1"
      },
      "execution_count": null,
      "outputs": []
    },
    {
      "cell_type": "code",
      "source": [
        "# %% [markdown]\n",
        "# EfficientNet-B3 backbone with a small classifier head\n",
        "\n",
        "# %% [code]\n",
        "def build_efficientnet_b3(num_classes=NUM_CLASSES, pretrained=True, fine_tune_last_blocks=True):\n",
        "    model = timm.create_model(\"efficientnet_b3\", pretrained=pretrained, num_classes=num_classes)\n",
        "    # Timm's classifier is model.classifier; its in_features vary by arch\n",
        "    in_features = model.classifier.in_features\n",
        "    model.classifier = nn.Sequential(\n",
        "        nn.Dropout(0.4),\n",
        "        nn.Linear(in_features, num_classes)\n",
        "    )\n",
        "    if fine_tune_last_blocks:\n",
        "        # Freeze early layers to stabilize training on small data\n",
        "        for name, p in model.named_parameters():\n",
        "            p.requires_grad = False\n",
        "        # Unfreeze last stages + classifier\n",
        "        for name, p in model.named_parameters():\n",
        "            if any(k in name for k in [\"blocks.6\", \"blocks.5\", \"conv_head\", \"bn2\", \"classifier\"]):\n",
        "                p.requires_grad = True\n",
        "    return model\n",
        "\n",
        "model = build_efficientnet_b3()\n",
        "model = model.to(DEVICE)\n",
        "\n",
        "# Train knobs\n",
        "LR  = 1e-4\n",
        "WD  = 1e-4\n",
        "EPOCHS = 12\n",
        "\n",
        "optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=WD)\n",
        "scheduler = ReduceLROnPlateau(optimizer, mode=\"max\", factor=0.5, patience=2, verbose=True)\n",
        "criterion  = nn.CrossEntropyLoss(weight=cls_weights)\n",
        "sum(p.numel() for p in model.parameters() if p.requires_grad), DEVICE\n"
      ],
      "metadata": {
        "id": "mgLvba0SW-qj"
      },
      "execution_count": null,
      "outputs": []
    },
    {
      "cell_type": "code",
      "source": [
        "# %% [markdown]\n",
        "# Train EfficientNet-B3 (last blocks) and track best model by macro-F1\n",
        "\n",
        "# %% [code]\n",
        "def evaluate_model(model, loader):\n",
        "    model.eval()\n",
        "    preds, targs = [], []\n",
        "    with torch.no_grad():\n",
        "        for x, y in loader:\n",
        "            x = x.to(DEVICE); y = y.to(DEVICE)\n",
        "            logits = model(x)\n",
        "            pred = logits.argmax(1)\n",
        "            preds.append(pred.cpu().numpy())\n",
        "            targs.append(y.cpu().numpy())\n",
        "    y_pred = np.concatenate(preds)\n",
        "    y_true = np.concatenate(targs)\n",
        "    acc  = accuracy_score(y_true, y_pred)\n",
        "    f1m  = f1_score(y_true, y_pred, average=\"macro\")\n",
        "    qwk  = cohen_kappa_score(y_true, y_pred, weights=\"quadratic\")\n",
        "    off1 = (np.abs(y_pred - y_true) <= 1).mean()\n",
        "    return acc, f1m, qwk, off1, y_true, y_pred\n",
        "\n",
        "best_f1 = -1.0\n",
        "BEST_PATH = \"/content/best_efficientnet_b3.pt\"\n",
        "\n",
        "for epoch in range(1, EPOCHS+1):\n",
        "    model.train()\n",
        "    running = 0.0\n",
        "    for x, y in dl_train:\n",
        "        x = x.to(DEVICE); y = y.to(DEVICE)\n",
        "        optimizer.zero_grad()\n",
        "        logits = model(x)\n",
        "        loss = criterion(logits, y)\n",
        "        loss.backward()\n",
        "        optimizer.step()\n",
        "        running += loss.item() * x.size(0)\n",
        "    tr_loss = running / len(ds_train)\n",
        "\n",
        "    acc, f1m, qwk, off1, y_true, y_pred = evaluate_model(model, dl_test)\n",
        "    scheduler.step(f1m)\n",
        "\n",
        "    print(f\"[{epoch:02d}] loss={tr_loss:.4f} | acc={acc:.3f} f1={f1m:.3f} qwk={qwk:.3f} off±1={off1:.3f}\")\n",
        "    if f1m > best_f1:\n",
        "        best_f1 = f1m\n",
        "        torch.save(model.state_dict(), BEST_PATH)\n",
        "        print(\"  → saved best\")\n",
        "\n",
        "# Load best for downstream\n",
        "model.load_state_dict(torch.load(BEST_PATH, map_location=DEVICE))\n"
      ],
      "metadata": {
        "id": "uZ1bA0kSW_vB"
      },
      "execution_count": null,
      "outputs": []
    },
    {
      "cell_type": "code",
      "source": [
        "# %% [markdown]\n",
        "# Turn the backbone into a feature extractor (1536-D for EfficientNet-B3)\n",
        "\n",
        "# %% [code]\n",
        "# Build a copy with classifier removed to get penultimate features\n",
        "embedder = timm.create_model(\"efficientnet_b3\", pretrained=False, num_classes=NUM_CLASSES)\n",
        "in_features = embedder.classifier.in_features\n",
        "# Replace classifier with identity to get GAP features\n",
        "embedder.classifier = nn.Identity()\n",
        "# Copy weights except classifier head\n",
        "state = model.state_dict()\n",
        "state = {k: v for k, v in state.items() if not k.startswith(\"classifier\")}\n",
        "missing, unexpected = embedder.load_state_dict(state, strict=False)\n",
        "embedder = embedder.to(DEVICE).eval()\n",
        "\n",
        "def extract_embeddings(model, loader):\n",
        "    feats, labels = [], []\n",
        "    with torch.no_grad():\n",
        "        for x, y in loader:\n",
        "            x = x.to(DEVICE)\n",
        "            f = model(x)  # (B, 1536)\n",
        "            feats.append(f.cpu().numpy())\n",
        "            labels.append(y.numpy())\n",
        "    return np.concatenate(feats), np.concatenate(labels)\n",
        "\n",
        "X_train_1536, y_train = extract_embeddings(embedder, dl_train)\n",
        "X_test_1536,  y_test  = extract_embeddings(embedder, dl_test)\n",
        "X_train_1536.shape, X_test_1536.shape\n"
      ],
      "metadata": {
        "id": "TSoP9HGJXDvP"
      },
      "execution_count": null,
      "outputs": []
    },
    {
      "cell_type": "code",
      "source": [
        "# %% [markdown]\n",
        "# Baseline: RBF SVM on PCA-256 of CNN features\n",
        "\n",
        "# %% [code]\n",
        "scaler = StandardScaler(with_mean=True, with_std=True)\n",
        "Xtr_std = scaler.fit_transform(X_train_1536)\n",
        "Xte_std = scaler.transform(X_test_1536)\n",
        "\n",
        "pca256 = PCA(n_components=256, random_state=SEED)\n",
        "Xtr_256 = pca256.fit_transform(Xtr_std)\n",
        "Xte_256 = pca256.transform(Xte_std)\n",
        "\n",
        "svm_baseline = SVC(kernel=\"rbf\", C=10.0, gamma=\"scale\", class_weight=\"balanced\", probability=False, random_state=SEED)\n",
        "svm_baseline.fit(Xtr_256, y_train)\n",
        "svm_pred = svm_baseline.predict(Xte_256)\n",
        "\n",
        "svm_acc  = accuracy_score(y_test, svm_pred)\n",
        "svm_f1   = f1_score(y_test, svm_pred, average=\"macro\")\n",
        "svm_qwk  = cohen_kappa_score(y_test, svm_pred, weights=\"quadratic\")\n",
        "svm_off1 = (np.abs(svm_pred - y_test) <= 1).mean()\n",
        "print(f\"SVM (RBF, PCA-256) → acc={svm_acc:.3f} f1={svm_f1:.3f} qwk={svm_qwk:.3f} off±1={svm_off1:.3f}\")\n"
      ],
      "metadata": {
        "id": "SW0fIBECXFjY"
      },
      "execution_count": null,
      "outputs": []
    },
    {
      "cell_type": "code",
      "source": [
        "# %% [markdown]\n",
        "# MLP on PCA-256 features (strong classical head)\n",
        "\n",
        "# %% [code]\n",
        "class MLPHead(nn.Module):\n",
        "    def __init__(self, in_dim=256, num_classes=NUM_CLASSES, p=0.4):\n",
        "        super().__init__()\n",
        "        self.net = nn.Sequential(\n",
        "            nn.Linear(in_dim, 512),\n",
        "            nn.BatchNorm1d(512), nn.GELU(), nn.Dropout(p),\n",
        "            nn.Linear(512, 128),\n",
        "            nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(p),\n",
        "            nn.Linear(128, num_classes)\n",
        "        )\n",
        "    def forward(self, x): return self.net(x)\n",
        "\n",
        "def train_mlp(Xtr, ytr, Xte, yte, epochs=25, lr=1e-3):\n",
        "    Xtr_t = torch.tensor(Xtr, dtype=torch.float32).to(DEVICE)\n",
        "    ytr_t = torch.tensor(ytr, dtype=torch.long).to(DEVICE)\n",
        "    Xte_t = torch.tensor(Xte, dtype=torch.float32).to(DEVICE)\n",
        "    yte_t = torch.tensor(yte, dtype=torch.long).to(DEVICE)\n",
        "\n",
        "    ds_tr = torch.utils.data.TensorDataset(Xtr_t, ytr_t)\n",
        "    dl_tr = DataLoader(ds_tr, batch_size=64, shuffle=True)\n",
        "\n",
        "    model = MLPHead(in_dim=Xtr.shape[1]).to(DEVICE)\n",
        "    opt = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)\n",
        "    crit = nn.CrossEntropyLoss(weight=cls_weights)\n",
        "    sch  = ReduceLROnPlateau(opt, mode=\"max\", factor=0.5, patience=3, verbose=False)\n",
        "\n",
        "    best_f1, best_state = -1, None\n",
        "    for ep in range(1, epochs+1):\n",
        "        model.train()\n",
        "        for xb, yb in dl_tr:\n",
        "            opt.zero_grad()\n",
        "            logits = model(xb)\n",
        "            loss = crit(logits, yb)\n",
        "            loss.backward(); opt.step()\n",
        "        # eval\n",
        "        model.eval()\n",
        "        with torch.no_grad():\n",
        "            logits = model(Xte_t)\n",
        "            pred = logits.argmax(1).cpu().numpy()\n",
        "        f1m = f1_score(yte, pred, average=\"macro\")\n",
        "        sch.step(f1m)\n",
        "        if f1m > best_f1:\n",
        "            best_f1 = f1m\n",
        "            best_state = model.state_dict()\n",
        "        print(f\"[MLP ep {ep:02d}] f1={f1m:.3f}\")\n",
        "    model.load_state_dict(best_state)\n",
        "    model.eval()\n",
        "    with torch.no_grad():\n",
        "        logits = model(Xte_t)\n",
        "        pred = logits.argmax(1).cpu().numpy()\n",
        "    acc = accuracy_score(yte, pred)\n",
        "    qwk = cohen_kappa_score(yte, pred, weights=\"quadratic\")\n",
        "    off1 = (np.abs(pred - yte) <= 1).mean()\n",
        "    return model, {\"acc\": acc, \"f1\": best_f1, \"qwk\": qwk, \"off1\": off1}, pred\n",
        "\n",
        "mlp_model, mlp_metrics, mlp_pred = train_mlp(Xtr_256, y_train, Xte_256, y_test, epochs=25, lr=1e-3)\n",
        "print(\"MLP →\", mlp_metrics)\n"
      ],
      "metadata": {
        "id": "izjt2-7YXGJa"
      },
      "execution_count": null,
      "outputs": []
    },
    {
      "cell_type": "code",
      "source": [
        "# %% [markdown]\n",
        "# Learn an 8-D supervised embedding (often better than raw PCA for QSVM)\n",
        "\n",
        "# %% [code]\n",
        "class Bridge8(nn.Module):\n",
        "    def __init__(self, in_dim=1536, out_dim=8, p=0.2):\n",
        "        super().__init__()\n",
        "        self.net = nn.Sequential(\n",
        "            nn.Linear(in_dim, 256),\n",
        "            nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(p),\n",
        "            nn.Linear(256, out_dim)\n",
        "        )\n",
        "        self.head = nn.Linear(out_dim, NUM_CLASSES)\n",
        "\n",
        "    def forward(self, x):\n",
        "        z = self.net(x)\n",
        "        logits = self.head(z)\n",
        "        return z, logits\n",
        "\n",
        "def train_bridge8(Xtr_1536, ytr, Xte_1536, yte, epochs=15, lr=1e-3):\n",
        "    Xtr = torch.tensor(Xtr_1536, dtype=torch.float32).to(DEVICE)\n",
        "    Xte = torch.tensor(Xte_1536, dtype=torch.float32).to(DEVICE)\n",
        "    ytr_t = torch.tensor(ytr, dtype=torch.long).to(DEVICE)\n",
        "    yte_t = torch.tensor(yte, dtype=torch.long).to(DEVICE)\n",
        "\n",
        "    ds_tr = torch.utils.data.TensorDataset(Xtr, ytr_t)\n",
        "    dl_tr = DataLoader(ds_tr, batch_size=64, shuffle=True)\n",
        "\n",
        "    model = Bridge8(in_dim=Xtr_1536.shape[1], out_dim=8).to(DEVICE)\n",
        "    opt = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)\n",
        "    crit = nn.CrossEntropyLoss(weight=cls_weights)\n",
        "    sch  = ReduceLROnPlateau(opt, mode=\"max\", factor=0.5, patience=2, verbose=False)\n",
        "\n",
        "    best_f1, best_state = -1, None\n",
        "    for ep in range(1, epochs+1):\n",
        "        model.train()\n",
        "        for xb, yb in dl_tr:\n",
        "            opt.zero_grad()\n",
        "            z, logits = model(xb)\n",
        "            loss = crit(logits, yb)\n",
        "            loss.backward(); opt.step()\n",
        "        # eval\n",
        "        model.eval()\n",
        "        with torch.no_grad():\n",
        "            zte, logitste = model(Xte)\n",
        "            pred = logitste.argmax(1).cpu().numpy()\n",
        "        f1m = f1_score(yte, pred, average=\"macro\")\n",
        "        sch.step(f1m)\n",
        "        if f1m > best_f1:\n",
        "            best_f1 = f1m\n",
        "            best_state = model.state_dict()\n",
        "        print(f\"[Bridge8 ep {ep:02d}] f1={f1m:.3f}\")\n",
        "    model.load_state_dict(best_state)\n",
        "    model.eval()\n",
        "    with torch.no_grad():\n",
        "        Ztr, _ = model(Xtr)\n",
        "        Zte, _ = model(Xte)\n",
        "    return model, Ztr.cpu().numpy(), Zte.cpu().numpy()\n",
        "\n",
        "bridge8, Xtr_8, Xte_8 = train_bridge8(X_train_1536, y_train, X_test_1536, y_test, epochs=15)\n",
        "Xtr_8.shape, Xte_8.shape\n"
      ],
      "metadata": {
        "id": "eJU2BVuWXKTj"
      },
      "execution_count": null,
      "outputs": []
    },
    {
      "cell_type": "code",
      "source": [
        "# %% [markdown]\n",
        "# QSVM with a shallow ZZ-style feature map on 8 qubits (quantum kernel)\n",
        "\n",
        "# %% [code]\n",
        "n_qubits = 8\n",
        "dev = qml.device(\"lightning.qubit\", wires=n_qubits)\n",
        "\n",
        "def feature_map(x):\n",
        "    # Angle encoding + linear entanglement (reps=2-ish)\n",
        "    # x expected shape: (8,)\n",
        "    for i in range(n_qubits):\n",
        "        qml.RY(x[i], wires=i)\n",
        "    for i in range(n_qubits-1):\n",
        "        qml.CZ(wires=[i, i+1])\n",
        "    for i in range(n_qubits):\n",
        "        qml.RZ(x[i], wires=i)\n",
        "    for i in range(n_qubits-1):\n",
        "        qml.CZ(wires=[i, i+1])\n",
        "\n",
        "@qml.qnode(dev, interface=None, diff_method=None)\n",
        "def kernel_circuit(x1, x2):\n",
        "    # Implements |<phi(x1)|phi(x2)>|^2 by preparing phi(x1), adjoint of phi(x2), then measuring all-zero projector.\n",
        "    # A simpler surrogate is state overlap via the adjoint trick:\n",
        "    qml.templates.AngleEmbedding(x1, wires=range(n_qubits), rotation='Y')  # not used if we define our own; kept as a placeholder\n",
        "    # Replace with our feature_map:\n",
        "    qml.apply(qml.ops.null_op)  # no-op to keep structure\n",
        "    # Encode x1\n",
        "    for i in range(n_qubits):\n",
        "        qml.RY(x1[i], wires=i)\n",
        "    for i in range(n_qubits-1):\n",
        "        qml.CZ(wires=[i, i+1])\n",
        "    for i in range(n_qubits):\n",
        "        qml.RZ(x1[i], wires=i)\n",
        "    for i in range(n_qubits-1):\n",
        "        qml.CZ(wires=[i, i+1])\n",
        "    # Adjoint of feature map for x2\n",
        "    qml.adjoint(feature_map)(x2)\n",
        "    # Probability of all-zeros state equals |<phi(x2)|phi(x1)>|^2\n",
        "    return qml.probs(wires=range(n_qubits))\n",
        "\n",
        "def quantum_kernel_matrix(A, B, scale_inputs=True):\n",
        "    # Scale inputs to a reasonable range for angles\n",
        "    if scale_inputs:\n",
        "        mu = A.mean(axis=0); sd = A.std(axis=0) + 1e-6\n",
        "        A_ = (A - mu) / sd\n",
        "        B_ = (B - mu) / sd\n",
        "        A_ = np.clip(A_, -3, 3) * (np.pi/2)\n",
        "        B_ = np.clip(B_, -3, 3) * (np.pi/2)\n",
        "    else:\n",
        "        A_, B_ = A, B\n",
        "    K = np.zeros((len(A_), len(B_)))\n",
        "    for i, a in enumerate(A_):\n",
        "        for j, b in enumerate(B_):\n",
        "            probs = kernel_circuit(a, b)\n",
        "            K[i, j] = probs[0]  # all-zero outcome\n",
        "    return K\n",
        "\n",
        "# Compute kernel matrices (this is the heaviest step; keep n_qubits small)\n",
        "K_train = quantum_kernel_matrix(Xtr_8, Xtr_8)\n",
        "K_test  = quantum_kernel_matrix(Xte_8, Xtr_8)\n",
        "\n",
        "qsvm = SVC(kernel=\"precomputed\", C=10.0, class_weight=\"balanced\")\n",
        "qsvm.fit(K_train, y_train)\n",
        "qsvm_pred = qsvm.predict(K_test)\n",
        "\n",
        "qsvm_acc  = accuracy_score(y_test, qsvm_pred)\n",
        "qsvm_f1   = f1_score(y_test, qsvm_pred, average=\"macro\")\n",
        "qsvm_qwk  = cohen_kappa_score(y_test, qsvm_pred, weights=\"quadratic\")\n",
        "qsvm_off1 = (np.abs(qsvm_pred - y_test) <= 1).mean()\n",
        "print(f\"QSVM (8-D) → acc={qsvm_acc:.3f} f1={qsvm_f1:.3f} qwk={qsvm_qwk:.3f} off±1={qsvm_off1:.3f}\")\n"
      ],
      "metadata": {
        "id": "5O7oECdvXMXm"
      },
      "execution_count": null,
      "outputs": []
    },
    {
      "cell_type": "code",
      "source": [
        "# %% [markdown]\n",
        "# Train a tiny QCNN to shape 8-D into a 4-D quantum embedding, then QSVM on 4-D.\n",
        "\n",
        "# %% [code]\n",
        "n_qubits_qcnn = 4\n",
        "dev_qcnn = qml.device(\"lightning.qubit\", wires=n_qubits_qcnn)\n",
        "\n",
        "@qml.qnode(dev_qcnn, interface=\"torch\")\n",
        "def qcnn_node(x4, theta):\n",
        "    # x4: (4,), theta: (2*4,)\n",
        "    # Encoder: angle-embed compressed features\n",
        "    for i in range(n_qubits_qcnn):\n",
        "        qml.RY(x4[i], wires=i)\n",
        "    # Block 1\n",
        "    for i in range(n_qubits_qcnn):\n",
        "        qml.RY(theta[i], wires=i)\n",
        "    for i in range(n_qubits_qcnn-1):\n",
        "        qml.CZ(wires=[i, i+1])\n",
        "    # Block 2\n",
        "    for i in range(n_qubits_qcnn):\n",
        "        qml.RY(theta[n_qubits_qcnn+i], wires=i)\n",
        "    for i in range(n_qubits_qcnn-1):\n",
        "        qml.CZ(wires=[i, i+1])\n",
        "\n",
        "    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits_qcnn)]\n",
        "\n",
        "class QCNNEmbed(nn.Module):\n",
        "    def __init__(self, in_dim=8, out_dim=4):\n",
        "        super().__init__()\n",
        "        self.reduce = nn.Linear(in_dim, out_dim)\n",
        "        nn.init.xavier_uniform_(self.reduce.weight)\n",
        "        self.theta = nn.Parameter(torch.zeros(2*n_qubits_qcnn))\n",
        "        self.head  = nn.Linear(out_dim, NUM_CLASSES)  # for shaping with CE\n",
        "\n",
        "    def forward(self, x8, return_embed=False):\n",
        "        # scale to angles\n",
        "        x4 = self.reduce(x8)\n",
        "        x4 = torch.tanh(x4) * (np.pi/2)\n",
        "        # call quantum node per-sample (small batches advised)\n",
        "        outs = []\n",
        "        for i in range(x4.shape[0]):\n",
        "            out = qcnn_node(x4[i], self.theta)\n",
        "            outs.append(out)\n",
        "        qemb = torch.stack(outs)  # (B, 4)\n",
        "        logits = self.head(qemb)\n",
        "        if return_embed:\n",
        "            return qemb, logits\n",
        "        return logits\n",
        "\n",
        "def train_qcnn_embed(Xtr8, ytr, Xte8, yte, epochs=10, lr=5e-4):\n",
        "    Xtr_t = torch.tensor(Xtr8, dtype=torch.float32).to(DEVICE)\n",
        "    ytr_t = torch.tensor(ytr, dtype=torch.long).to(DEVICE)\n",
        "    Xte_t = torch.tensor(Xte8, dtype=torch.float32).to(DEVICE)\n",
        "    yte_t = torch.tensor(yte, dtype=torch.long).to(DEVICE)\n",
        "\n",
        "    ds_tr = torch.utils.data.TensorDataset(Xtr_t, ytr_t)\n",
        "    dl_tr = DataLoader(ds_tr, batch_size=32, shuffle=True)\n",
        "\n",
        "    model = QCNNEmbed(in_dim=8, out_dim=4).to(DEVICE)\n",
        "    opt = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)\n",
        "    crit = nn.CrossEntropyLoss(weight=cls_weights)\n",
        "    sch  = ReduceLROnPlateau(opt, mode=\"max\", factor=0.5, patience=2, verbose=False)\n",
        "\n",
        "    best_f1, best_state = -1, None\n",
        "    for ep in range(1, epochs+1):\n",
        "        model.train()\n",
        "        for xb, yb in dl_tr:\n",
        "            opt.zero_grad()\n",
        "            logits = model(xb, return_embed=False)\n",
        "            loss = crit(logits, yb)\n",
        "            loss.backward(); opt.step()\n",
        "        # eval\n",
        "        model.eval()\n",
        "        with torch.no_grad():\n",
        "            qemb_te, logits_te = model(Xte_t, return_embed=True)\n",
        "            pred = logits_te.argmax(1).cpu().numpy()\n",
        "        f1m = f1_score(yte, pred, average=\"macro\")\n",
        "        sch.step(f1m)\n",
        "        if f1m > best_f1:\n",
        "            best_f1 = f1m\n",
        "            best_state = model.state_dict()\n",
        "        print(f\"[QCNN ep {ep:02d}] shaping-f1={f1m:.3f}\")\n",
        "    model.load_state_dict(best_state)\n",
        "    model.eval()\n",
        "    with torch.no_grad():\n",
        "        qemb_tr, _ = model(Xtr_t, return_embed=True)\n",
        "        qemb_te, _ = model(Xte_t, return_embed=True)\n",
        "    return model, qemb_tr.cpu().numpy(), qemb_te.cpu().numpy()\n",
        "\n",
        "qcnn_model, Xtr_4, Xte_4 = train_qcnn_embed(Xtr_8, y_train, Xte_8, y_test, epochs=10)\n",
        "Xtr_4.shape, Xte_4.shape\n"
      ],
      "metadata": {
        "id": "ch5qeEI5XOlx"
      },
      "execution_count": null,
      "outputs": []
    },
    {
      "cell_type": "code",
      "source": [
        "# %% [markdown]\n",
        "# Final quantum classifier: QSVM on 4-D QCNN embeddings\n",
        "\n",
        "# %% [code]\n",
        "n_qubits_4 = 4\n",
        "dev_k4 = qml.device(\"lightning.qubit\", wires=n_qubits_4)\n",
        "\n",
        "def feature_map_4(x):\n",
        "    for i in range(n_qubits_4):\n",
        "        qml.RY(x[i], wires=i)\n",
        "    for i in range(n_qubits_4-1):\n",
        "        qml.CZ(wires=[i, i+1])\n",
        "    for i in range(n_qubits_4):\n",
        "        qml.RZ(x[i], wires=i)\n",
        "\n",
        "@qml.qnode(dev_k4, interface=None, diff_method=None)\n",
        "def kernel_circuit_4(x1, x2):\n",
        "    # Encode x1\n",
        "    for i in range(n_qubits_4):\n",
        "        qml.RY(x1[i], wires=i)\n",
        "    for i in range(n_qubits_4-1):\n",
        "        qml.CZ(wires=[i, i+1])\n",
        "    for i in range(n_qubits_4):\n",
        "        qml.RZ(x1[i], wires=i)\n",
        "    # Adjoint of x2 map\n",
        "    qml.adjoint(feature_map_4)(x2)\n",
        "    return qml.probs(wires=range(n_qubits_4))\n",
        "\n",
        "def quantum_kernel_matrix_4(A, B):\n",
        "    # Inputs already roughly scaled (tanh*(pi/2) inside QCNN); keep as is\n",
        "    K = np.zeros((len(A), len(B)))\n",
        "    for i, a in enumerate(A):\n",
        "        for j, b in enumerate(B):\n",
        "            probs = kernel_circuit_4(a, b)\n",
        "            K[i, j] = probs[0]\n",
        "    return K\n",
        "\n",
        "K_train_4 = quantum_kernel_matrix_4(Xtr_4, Xtr_4)\n",
        "K_test_4  = quantum_kernel_matrix_4(Xte_4, Xtr_4)\n",
        "\n",
        "qsvm4 = SVC(kernel=\"precomputed\", C=10.0, class_weight=\"balanced\")\n",
        "qsvm4.fit(K_train_4, y_train)\n",
        "qsvm4_pred = qsvm4.predict(K_test_4)\n",
        "\n",
        "qsvm4_acc  = accuracy_score(y_test, qsvm4_pred)\n",
        "qsvm4_f1   = f1_score(y_test, qsvm4_pred, average=\"macro\")\n",
        "qsvm4_qwk  = cohen_kappa_score(y_test, qsvm4_pred, weights=\"quadratic\")\n",
        "qsvm4_off1 = (np.abs(qsvm4_pred - y_test) <= 1).mean()\n",
        "print(f\"QCNN-4 + QSVM → acc={qsvm4_acc:.3f} f1={qsvm4_f1:.3f} qwk={qsvm4_qwk:.3f} off±1={qsvm4_off1:.3f}\")\n"
      ],
      "metadata": {
        "id": "WCZ_ZevMXQ1C"
      },
      "execution_count": null,
      "outputs": []
    },
    {
      "cell_type": "code",
      "source": [
        "# %% [markdown]\n",
        "# Compare all models\n",
        "\n",
        "# %% [code]\n",
        "summary = pd.DataFrame([\n",
        "    {\"model\": \"CNN+SVM (RBF, PCA-256)\", \"acc\": svm_acc,  \"f1\": svm_f1,  \"qwk\": svm_qwk,  \"off±1\": svm_off1},\n",
        "    {\"model\": \"CNN+MLP (PCA-256)\",      \"acc\": mlp_metrics[\"acc\"], \"f1\": mlp_metrics[\"f1\"], \"qwk\": mlp_metrics[\"qwk\"], \"off±1\": mlp_metrics[\"off1\"]},\n",
        "    {\"model\": \"CNN+Bridge8 → QSVM\",     \"acc\": qsvm_acc, \"f1\": qsvm_f1, \"qwk\": qsvm_qwk, \"off±1\": qsvm_off1},\n",
        "    {\"model\": \"CNN+Bridge8 → QCNN-4 → QSVM\", \"acc\": qsvm4_acc, \"f1\": qsvm4_f1, \"qwk\": qsvm4_qwk, \"off±1\": qsvm4_off1},\n",
        "])\n",
        "summary\n"
      ],
      "metadata": {
        "id": "bX8tGxkWXS7Z"
      },
      "execution_count": null,
      "outputs": []
    }
  ]
}

In [ ]:
import os
import zipfile
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import StackingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
import xgboost as xgb

# ----------------------------
# 1️⃣ Configuration and unzip
# ----------------------------
CONFIG = {
    'IMG_SIZE': (224, 224),
    'DATA_DIR': '/content/diabeties',
    'ZIP_PATH': '/content/drive/MyDrive/diabeties.zip',
    'EXTRACT_PATH': '/content/',
    'BATCH_SIZE': 32,
    'RANDOM_STATE': 42,
    'CLASS_NAMES': {
        0: 'No DR',
        1: 'Mild',
        2: 'Moderate',
        3: 'Severe',
        4: 'Proliferative DR'
    }
}

# Unzip only if folder doesn't exist
if not os.path.exists(CONFIG['DATA_DIR']):
    with zipfile.ZipFile(CONFIG['ZIP_PATH'], 'r') as zip_ref:
        zip_ref.extractall(CONFIG['EXTRACT_PATH'])
    print("✅ Dataset extracted successfully!")
else:
    print("📂 Dataset already extracted.")

# ----------------------------
# 2️⃣ Load dataset (train, val, test)
# ----------------------------
img_size = CONFIG['IMG_SIZE']
batch_size = CONFIG['BATCH_SIZE']
data_dir = CONFIG['DATA_DIR']

train_ds = keras.utils.image_dataset_from_directory(
    os.path.join(data_dir, "train"),
    image_size=img_size,
    batch_size=batch_size,
    label_mode='int',
    shuffle=True,
    seed=CONFIG['RANDOM_STATE']
)

val_ds = keras.utils.image_dataset_from_directory(
    os.path.join(data_dir, "val"),
    image_size=img_size,
    batch_size=batch_size,
    label_mode='int',
    shuffle=False
)

test_ds = keras.utils.image_dataset_from_directory(
    os.path.join(data_dir, "test"),
    image_size=img_size,
    batch_size=batch_size,
    label_mode='int',
    shuffle=False
)

# Normalize and prefetch
normalization_layer = layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y)).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y)).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.map(lambda x, y: (normalization_layer(x), y)).prefetch(tf.data.AUTOTUNE)

# ----------------------------
# 3️⃣ Define CNN for feature extraction
# ----------------------------
inputs = keras.Input(shape=img_size + (3,))
x = layers.Conv2D(32, (3,3), activation='relu')(inputs)
x = layers.MaxPooling2D((2,2))(x)
x = layers.Conv2D(64, (3,3), activation='relu')(x)
x = layers.MaxPooling2D((2,2))(x)
x = layers.Conv2D(128, (3,3), activation='relu')(x)
x = layers.Flatten()(x)
feature_layer = layers.Dense(128, activation='relu', name='feature_layer')(x)
outputs = layers.Dense(5, activation='softmax')(feature_layer)

cnn_model = keras.Model(inputs=inputs, outputs=outputs)
cnn_model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
cnn_model.summary()

# ----------------------------
# 4️⃣ Train CNN
# ----------------------------
cnn_model.fit(train_ds, validation_data=val_ds, epochs=5)

# ----------------------------
# 5️⃣ Extract features from CNN
# ----------------------------
feature_extractor = keras.Model(
    inputs=cnn_model.input,
    outputs=cnn_model.get_layer("feature_layer").output
)

def extract_features(dataset):
    features_list, labels_list = [], []
    for batch_images, batch_labels in dataset:
        feats = feature_extractor(batch_images, training=False)
        features_list.append(feats.numpy())
        labels_list.append(batch_labels.numpy())
    X = np.concatenate(features_list, axis=0)
    y = np.concatenate(labels_list, axis=0)
    return X, y

X_train, y_train = extract_features(train_ds)
X_test, y_test = extract_features(test_ds)

# ----------------------------
# 6️⃣ Stack SVM + XGBoost classifier
# ----------------------------
# Scale features (SVM benefits from scaling)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define base learners
svm_model = SVC(kernel='rbf', C=10, gamma='scale', probability=True)
xgb_model = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=5,
    learning_rate=0.1,
    n_estimators=100,
    max_depth=6,
    random_state=CONFIG['RANDOM_STATE']
)

# Stacking ensemble with Logistic Regression as meta-classifier
stack_model = StackingClassifier(
    estimators=[('svm', svm_model), ('xgb', xgb_model)],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5
)

# Train stacked model
stack_model.fit(X_train_scaled, y_train)

# ----------------------------
# 7️⃣ Evaluate stacked model
# ----------------------------
y_pred = stack_model.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)
print(f"\n🎯 Stacked SVM + XGBoost Accuracy: {acc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=list(CONFIG['CLASS_NAMES'].values())))





In [ ]:
from google.colab import drive
import zipfile
import os

# Mount Google Drive
drive.mount('/content/drive')

# Path to ZIP file in Drive
zip_path = "/content/drive/My Drive/diabeties.zip"

# Extract ZIP contents
extract_path = "/content/extracted_files"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Extracted successfully to:", extract_path)


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Data generators (no augmentation)
train_gen = ImageDataGenerator(rescale=1./255)
val_gen = ImageDataGenerator(rescale=1./255)
test_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    '/content/extracted_files/diabeties/train',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

val_data = val_gen.flow_from_directory(
    '/content/extracted_files/diabeties/val',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

test_data = test_gen.flow_from_directory(
    '/content/extracted_files/diabeties/test',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Dropout(0.3),

    Conv2D(256, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Dropout(0.4),

    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(5, activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

lr_sched = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=40,
    callbacks=[lr_sched, early_stop]
)

loss, acc = model.evaluate(test_data)
print(f"✅ Test Accuracy: {acc*100:.2f}%")
